# V37: frozen conditional delivery
Private offline training-only related gate; if passed, full-host 8B CE; otherwise fixed V21 fallback. No hidden-label selection and no claimed score gain. Original V16 attribution retained.

In [ ]:
import time,signal
V37_STARTED=time.time()
V37_DEADLINE=V37_STARTED+31800
def v37_timeout(signum,frame):raise TimeoutError('V37 global runtime budget exhausted')
signal.signal(signal.SIGALRM,v37_timeout)
signal.alarm(31800)


In [ ]:
import subprocess,sys,importlib.metadata
from packaging.version import Version
try: version=importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError: version=None
if version is not None and Version(version)<Version('0.16.0'):
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)

import glob,hashlib
from pathlib import Path
wheels=glob.glob('/kaggle/input/**/bitsandbytes-0.50.0-py3-none-manylinux_2_24_x86_64.whl',recursive=True)
assert len(wheels)==1,wheels
assert hashlib.sha256(Path(wheels[0]).read_bytes()).hexdigest()=='173d137610468bec9cddbaa2e049254e97792657ab984e3e737bec1772c1668c'
subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--force-reinstall',wheels[0]],check=True)
assert importlib.metadata.version('bitsandbytes')=='0.50.0'


In [ ]:
import os,sys,json,hashlib,subprocess,time,shutil
from pathlib import Path
V37_ROOT=Path.cwd()
def remaining_budget():
    remaining=V37_DEADLINE-time.time()
    assert remaining>0,'V37 global runtime exhausted'
    return remaining
def run_stage(command,check=True,timeout=None,**kwargs):
    import psutil
    process=subprocess.Popen(command,**kwargs)
    tracked={};started=time.monotonic()
    try:
        while True:
            try:
                for child in psutil.Process(process.pid).children(recursive=True):tracked[child.pid]=child
            except psutil.NoSuchProcess:pass
            remaining=None if timeout is None else timeout-(time.monotonic()-started)
            if remaining is not None and remaining<=0:raise subprocess.TimeoutExpired(command,timeout)
            try:
                status=process.wait(timeout=.5 if remaining is None else min(.5,remaining));break
            except subprocess.TimeoutExpired:continue
        if check and status:raise subprocess.CalledProcessError(status,command)
        return status
    except BaseException:
        # Capture descendants before killing the runner, including separate sessions.
        try:
            for child in psutil.Process(process.pid).children(recursive=True):tracked[child.pid]=child
        except psutil.NoSuchProcess:pass
        descendants=list(tracked.values())
        for child in reversed(descendants):
            try:child.terminate()
            except psutil.NoSuchProcess:pass
        if process.poll() is None:process.terminate()
        _,alive=psutil.wait_procs(descendants,timeout=3)
        for child in alive:
            try:child.kill()
            except psutil.NoSuchProcess:pass
        if process.poll() is None:process.kill()
        process.wait()
        psutil.wait_procs(alive,timeout=3)
        raise
manifest={'files': [{'path': 'artifacts/baseline-v1/folds.csv', 'bytes': 53998, 'sha256': 'c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508'}, {'path': 'artifacts/improvement-v17/selected-pool-ids.csv', 'bytes': 645393, 'sha256': 'ef130cdd1f390ed26b5948e3d05cd294e4f3bede6f2407456f40fdb9fa016c98'}, {'path': 'artifacts/improvement-v19/kaggle-output/supcon_final_validation.npz', 'bytes': 301551, 'sha256': 'ee68e3fdbe3c4b86f5a583630df4cfb2b1d78225fcc6e79ac8936b70a5753d6e'}, {'path': 'artifacts/improvement-v25/kaggle-output/ce_final_validation.npz', 'bytes': 301444, 'sha256': 'f44631e99727578ba6083ea6d6ee4ce184ea2a0e7d61521a561bced98d503f05'}, {'path': 'artifacts/improvement-v32/oof.npz', 'bytes': 391346, 'sha256': '39285926a06a9cd469cb774c1c9c68a2fdb7ba707efe1145c7ca579d08271543'}, {'path': 'artifacts/improvement-v34/evaluation-inputs.json', 'bytes': 938, 'sha256': '1b53403cd66deafa08075a1a6400cb9b9c45033b2e891d9a15ce110aa8458b41'}, {'path': 'artifacts/improvement-v34/frozen-plan.json', 'bytes': 26486, 'sha256': 'ad2eca217d73e4da241f6f1fdf9c2c3da34288af24d254c7b4db219e645878cb'}, {'path': 'artifacts/improvement-v34/primary-audit.json', 'bytes': 3917, 'sha256': 'c08145212c44ab6b3b6e0064db5178ece7a01ec6005477ec97c356f467352ead'}, {'path': 'artifacts/improvement-v34/primary-oof.npz', 'bytes': 769422, 'sha256': '6609dcecf4cd18bc5fbb901794feb90a7e2e9f78485f572942820b30258c4021'}, {'path': 'artifacts/improvement-v34/related/frozen-plan.json', 'bytes': 25679, 'sha256': '7e0473a7e450796ee03e00353bf3d776fd5aacd03062646dbd058bf593bbc837'}, {'path': 'artifacts/improvement-v34/remote-verification.json', 'bytes': 438, 'sha256': 'ed6b684ad5536fef4e8dfa2c1c009530d3c4c96ce581ff8a490955c4088bc4e2'}, {'path': 'artifacts/improvement-v4/robustness/folds.csv', 'bytes': 105290, 'sha256': '090c42e940fcfd380d821e2e7a6414605e301f826e5e6e29ad678be5643eee32'}, {'path': 'configs/name-that-llm/v32.json', 'bytes': 1378, 'sha256': 'f59bf0c7cb7a8d7d7995f07ee5a8aa8ac6924391338581c08698bc33b2104366'}, {'path': 'configs/name-that-llm/v34.json', 'bytes': 5171, 'sha256': 'c5fde950434c1dd604a3233d9b989790d9af105de9c3d732172c73f3539a35db'}, {'path': 'data/name-that-llm/external/pool_v4.parquet', 'bytes': 35403026, 'sha256': '37750e899ff6634c147daa7888bfed048e92d3fb1381fe5d21f93bf72a84e8ec'}, {'path': 'data/name-that-llm/train.csv', 'bytes': 2139418, 'sha256': '40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707'}, {'path': 'scripts/audit_full_v34.py', 'bytes': 4163, 'sha256': '91f4b3132510c3cf5788588e5a5d306120d03e2933ea6c6659a55a9e722afa20'}, {'path': 'scripts/build_full_v34.py', 'bytes': 4694, 'sha256': 'ac88cddb59b033227ca9697695fa524fbc193d37d06984247df62830901749ad'}, {'path': 'scripts/build_related_v34.py', 'bytes': 6367, 'sha256': '68aadd371a3aa7b58ee3de2b0b92409cfc02eca8a99ec632f80fbbc169d90d39'}, {'path': 'scripts/build_training_v34.py', 'bytes': 12185, 'sha256': '31088596f001fc5411ddb3dd2e444ce2964c1a5f9707485aedd3a9ecc9df2545'}, {'path': 'scripts/component_v26.py', 'bytes': 6728, 'sha256': '103ed079e352581157b6707c3518b80a2bedf4760d6e46ed1922a96f0d43618b'}, {'path': 'scripts/diagnose_balance_transfer.py', 'bytes': 5013, 'sha256': '89a375811336593d0388743856e8161fc25775694c18fbf9174bc62102b828eb'}, {'path': 'scripts/evaluate_related_v34.py', 'bytes': 7606, 'sha256': 'bfa997d7fd072ac70dae69e9c21f9620b98852d775563cc84ca941cf2e5f9de9'}, {'path': 'scripts/evaluate_v34.py', 'bytes': 12535, 'sha256': '6fc37a19f0bc04124476c86c567070418c6c3bff60ea9f11c4a5798cfada5042'}, {'path': 'scripts/prepare_likelihood_v33.py', 'bytes': 2105, 'sha256': '9517acff96c5f73c2838a3cc06bdc20aadeff9e14c801b460e3d6cd28c222838'}, {'path': 'scripts/train_adapter_v25.py', 'bytes': 9868, 'sha256': '12c5b7a5f7fb1384dbbd864134b281fe8d3699b61682a598c79aa2b62b722f6d'}, {'path': 'scripts/train_full_v26.py', 'bytes': 6832, 'sha256': '76760f88e291cf4fde208481791227af901febc7c309afcc6ad50c5693b0b9ca'}], 'private': True}
marker='artifacts/improvement-v34/evaluation-inputs.json'
marker_hash=next(e['sha256'] for e in manifest['files'] if e['path']==marker)
auxiliary=[p.parents[2] for p in Path('/kaggle/input').rglob('evaluation-inputs.json')
           if hashlib.sha256(p.read_bytes()).hexdigest()==marker_hash and p.as_posix().endswith(marker)]
assert len(auxiliary)==1,auxiliary
V37_AUXILIARY=auxiliary[0]
for entry in manifest['files']:
    source=V37_AUXILIARY/entry['path'];assert source.stat().st_size==entry['bytes']
    assert hashlib.sha256(source.read_bytes()).hexdigest()==entry['sha256'],str(source)
    destination=V37_ROOT/entry['path'];assert not destination.exists()
    destination.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(source,destination)
def filtered_inputs(source,destination):
    destination.mkdir()
    for child in source.iterdir():
        if child==V37_AUXILIARY:continue
        target=destination/child.name
        # pathlib.rglob does not recurse directory symlinks: mirror directories,
        # and symlink only individual files without copying large model weights.
        if child.is_dir():filtered_inputs(child,target)
        else:target.symlink_to(child)
V37_TRAINING_INPUT=V37_ROOT/'v37-training-inputs'
filtered_inputs(Path('/kaggle/input'),V37_TRAINING_INPUT)
assert len(list(V37_TRAINING_INPUT.rglob('train.csv')))==1
def linked_output(experiment,destination):
    candidates=[]
    for p in Path('/kaggle/input').rglob('plan.json'):
        try: v=json.loads(p.read_text())
        except (ValueError,OSError):continue
        if v.get('experiment')==experiment and (experiment!='improvement-v34' or p.parent.joinpath('summary.json').exists()):
            candidates.append(p.parent)
    assert len(candidates)==1,(experiment,candidates)
    dst=Path(destination);dst.parent.mkdir(parents=True,exist_ok=True)
    if not dst.exists():dst.symlink_to(candidates[0],target_is_directory=True)
    else:
        # Pilot NPZ is already staged here; overlay attached immutable outputs.
        for source in candidates[0].iterdir():
            target=dst/source.name
            if target.exists():
                assert target.is_file() and source.is_file()
                assert hashlib.sha256(target.read_bytes()).hexdigest()==hashlib.sha256(source.read_bytes()).hexdigest()
            else:target.symlink_to(source,target_is_directory=source.is_dir())
linked_output('improvement-v18','artifacts/improvement-v18/kaggle-output')
linked_output('improvement-v25','artifacts/improvement-v25/kaggle-output')
linked_output('improvement-v34','artifacts/improvement-v34/kaggle-output')
related_code='import os, sys, subprocess, json, hashlib, time, signal\nfrom pathlib import Path\nfor key in [\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\']: os.environ[key]=\'2\'\nos.environ[\'HF_HUB_OFFLINE\']=\'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\']=\'1\'\nos.environ[\'TOKENIZERS_PARALLELISM\']=\'false\'\nPath(\'run_confirmation_arm_v34.py\').write_text(\'"""Run one fixed recipe on independent host folds; each fit resumes external-only."""\\nimport hashlib\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport subprocess\\nimport sys\\nimport time\\n\\nfrom prepare_likelihood_v33 import prepare\\n\\n\\ndef main():\\n    root = Path.cwd()\\n    plan = json.loads((root/\\\'plan.json\\\').read_text())\\n    arm = sys.argv[1]\\n    spec = plan[\\\'arms\\\'][arm]\\n    records = []\\n    for fold in plan[\\\'confirmation_folds\\\']:\\n        folder = root/arm/f\\\'fold-{fold}\\\'\\n        assert not folder.exists(), \\\'Never overwrite an existing fold run\\\'\\n        folder.mkdir(parents=True)\\n        params = dict(spec[\\\'parameters\\\'])\\n        params.update(experiment=\\\'improvement-v34\\\', pilot_fold=fold,\\n                      labels=plan[\\\'labels\\\'], model_dir=spec[\\\'model_dir\\\'],\\n                      checkpoint=spec[\\\'checkpoint\\\'], checkpoint_sha256=spec[\\\'checkpoint_sha256\\\'],\\n                      confirmation_split=f\\\'related-{fold}\\\', confirmation_fold=fold,\\n                      host_sha256=plan[\\\'host_sha256\\\'], **plan[\\\'fold_sizes\\\'][str(fold)])\\n        prepare(plan[\\\'host_path\\\'], plan[\\\'fold_by_prompt\\\'], params, folder)\\n        (folder/\\\'training-plan.json\\\').write_text(json.dumps(params,indent=2))\\n        (folder/\\\'plan.json\\\').write_text(json.dumps(params,indent=2))\\n        worker = root/spec[\\\'training_worker\\\']\\n        env = dict(os.environ,V18_VARIANT=spec[\\\'variant\\\'])\\n        def run(path, environment):\\n            remaining = plan[\\\'deadline_unix\\\']-time.time()\\n            if remaining <= 0:\\n                raise TimeoutError(\\\'Confirmation notebook wall budget exhausted\\\')\\n            subprocess.run([sys.executable,\\\'-u\\\',str(path)],cwd=folder,env=environment,\\n                           check=True,timeout=remaining)\\n        run(worker,env)\\n        training = json.loads((folder/spec[\\\'variant\\\']/\\\'result.json\\\').read_text())\\n        assert training[\\\'completed\\\'] and training[\\\'validation_rows\\\']==params[\\\'validation_rows\\\']\\n        readout = dict(params)\\n        readout.update(spec[\\\'readout\\\'])\\n        readout.update(host_path=plan[\\\'host_path\\\'],upstream_path=str(folder))\\n        weight = folder/spec[\\\'variant\\\']/\\\'final/adapter/adapter_model.safetensors\\\'\\n        readout[\\\'adapter_sha256\\\'] = {spec[\\\'readout_variant\\\']:hashlib.sha256(weight.read_bytes()).hexdigest()}\\n        (folder/\\\'readout-plan.json\\\').write_text(json.dumps(readout,indent=2))\\n        (folder/\\\'plan.json\\\').write_text(json.dumps(readout,indent=2))\\n        try:\\n            run(root/spec[\\\'readout_worker\\\'],dict(env,V19_VARIANTS=json.dumps([spec[\\\'readout_variant\\\']])))\\n        finally:\\n            (folder/\\\'plan.json\\\').write_text(json.dumps(params,indent=2))\\n        result = json.loads((folder/(spec[\\\'readout_variant\\\']+\\\'_result.json\\\')).read_text())\\n        record = {\\\'arm\\\':arm,\\\'fold\\\':fold,\\\'training\\\':training,\\\'readout\\\':result}\\n        records.append(record)\\n        with (root/(arm+\\\'-progress.jsonl\\\')).open(\\\'a\\\') as f:\\n            f.write(json.dumps(record)+\\\'\\\\n\\\')\\n        print(json.dumps(record),flush=True)\\n    (root/(arm+\\\'-complete.json\\\')).write_text(json.dumps({\\\'completed\\\':True,\\\'folds\\\':records},indent=2))\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    main()\\n\')\nPath(\'prepare_likelihood_v33.py\').write_text(\'"""Prepare one fixed host holdout, retaining related-prompt group boundaries."""\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nimport pandas as pd\\n\\n\\ndef prepare(host_path, split_by_prompt, plan, output):\\n    host_path = Path(host_path)\\n    assert hashlib.sha256(host_path.read_bytes()).hexdigest() == plan[\\\'host_sha256\\\']\\n    host = pd.read_csv(host_path, dtype={\\\'id\\\': str, \\\'prompt_id\\\': str})\\n    assert len(host) == 2699 and host.id.is_unique\\n    assert set(host.prompt_id) == set(split_by_prompt)\\n    labels = sorted(host.model.unique())\\n    assert labels == plan[\\\'labels\\\'] and len(labels) == 18\\n    host[\\\'group\\\'] = host.prompt_id.map(lambda p: str(split_by_prompt[p][\\\'group\\\']))\\n    assigned = host.prompt_id.map(lambda p: int(split_by_prompt[p][\\\'fold\\\']))\\n    assert host.assign(fold=assigned).groupby(\\\'group\\\').fold.nunique().max() == 1\\n    heldout = assigned == plan[\\\'confirmation_fold\\\']\\n    host[\\\'stage\\\'] = \\\'host\\\'\\n    host[\\\'label\\\'] = host.model.map({name: i for i, name in enumerate(labels)}).astype(int)\\n    train = host.loc[~heldout].reset_index(drop=True)\\n    valid = host.loc[heldout].reset_index(drop=True)\\n    assert len(train) == plan[\\\'training_rows\\\'] and len(valid) == plan[\\\'validation_rows\\\']\\n    assert set(train.group).isdisjoint(valid.group)\\n    assert set(train.prompt_id).isdisjoint(valid.prompt_id)\\n    output = Path(output)\\n    output.mkdir(parents=True, exist_ok=True)\\n    paths = [output / name for name in [\\\'training_frame.parquet\\\', \\\'validation_frame.parquet\\\']]\\n    assert not any(p.exists() for p in paths), \\\'Existing prepared frames must not be overwritten\\\'\\n    for path, frame in zip(paths, [train, valid]):\\n        frame.to_parquet(path, index=False)\\n    audit = {\\\'split\\\': plan[\\\'confirmation_split\\\'], \\\'training_rows\\\': len(train),\\n             \\\'validation_rows\\\': len(valid), \\\'training_groups\\\': train.group.nunique(),\\n             \\\'validation_groups\\\': valid.group.nunique(), \\\'labels\\\': labels,\\n             \\\'sha256\\\': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}}\\n    (output / \\\'data-audit.json\\\').write_text(json.dumps(audit, indent=2))\\n    return audit\\n\')\nPath(\'8b_ce_worker.py\').write_text(\'"""Standalone GPU worker for the frozen v18 paired-objective pilot."""\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport time\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\n\\ndef supcon_loss(features, labels, temperature=0.1):\\n    """Mean log-probability of same-class positives, excluding self pairs."""\\n    z = F.normalize(features.float(), dim=1)\\n    logits = z @ z.T / temperature\\n    diagonal = torch.eye(len(z), dtype=torch.bool, device=z.device)\\n    positives = labels[:, None].eq(labels[None, :]) & ~diagonal\\n    counts = positives.sum(1)\\n    if not bool((counts > 0).all()):\\n        raise ValueError("Every contrastive anchor must have a distinct positive.")\\n    denominator = torch.logsumexp(logits.masked_fill(diagonal, -torch.inf), dim=1)\\n    return -((logits - denominator[:, None]).masked_fill(~positives, 0).sum(1) / counts).mean()\\n\\n\\ndef paired_batches(frame, steps, seed):\\n    """Two authors, two different prompts each; identical schedule for both arms."""\\n    rng = np.random.default_rng(seed)\\n    groups = {label: {p: sub.index.to_numpy() for p, sub in group.groupby("group")}\\n              for label, group in frame.groupby("label")}\\n    assert all(len(g) >= 2 for g in groups.values())\\n    labels = sorted(groups)\\n    for _ in range(steps):\\n        selected = []\\n        for label in rng.choice(labels, 2, replace=False):\\n            for prompt in rng.choice(list(groups[label]), 2, replace=False):\\n                selected.append(int(rng.choice(groups[label][prompt])))\\n        yield selected\\n\\n\\nclass AttributionModel(nn.Module):\\n    def __init__(self, model_dir, plan):\\n        super().__init__()\\n        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\\n        from transformers import AutoModel, BitsAndBytesConfig\\n        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\\n        backbone = AutoModel.from_pretrained(model_dir, torch_dtype=torch.float16,\\n            quantization_config=quant, device_map={"":0},\\n            local_files_only=True, attn_implementation="sdpa")\\n        assert backbone.config.hidden_size == 4096 and backbone.config.num_hidden_layers == 36\\n        backbone = prepare_model_for_kbit_training(backbone,\\n            gradient_checkpointing_kwargs={"use_reentrant":False})\\n        backbone.config.use_cache = False\\n        backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\\n        self.encoder = get_peft_model(backbone, LoraConfig(r=plan["lora_rank"],\\n            lora_alpha=plan["lora_alpha"], target_modules=plan["target_modules"], lora_dropout=0.0,\\n            bias="none"))\\n        width = backbone.config.hidden_size\\n        self.norm = nn.LayerNorm(width)\\n        self.head = nn.Linear(width, 18)\\n        self.projection = nn.Sequential(nn.Linear(width, 256), nn.GELU(), nn.Linear(256, 128))\\n\\n    def forward(self, batch):\\n        hidden = self.encoder(**batch, use_cache=False).last_hidden_state.float()\\n        mask = batch["attention_mask"].clone()\\n        mask[:, 0] = 0\\n        pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)\\n        features = self.norm(pooled)\\n        return self.head(features), self.projection(features)\\n\\n\\ndef main():\\n    import pandas as pd\\n    from transformers import AutoTokenizer\\n    import transformers, peft\\n    plan = json.loads(Path("plan.json").read_text())\\n    variant = os.environ["V18_VARIANT"]\\n    assert variant in plan["variants"]\\n    out = Path(variant)\\n    out.mkdir(exist_ok=True)\\n    started = time.monotonic()\\n    def log(**kw):\\n        record = dict(variant=variant, seconds=round(time.monotonic()-started, 2), **kw)\\n        print(json.dumps(record), flush=True)\\n        with (out / "progress.jsonl").open("a") as f:\\n            f.write(json.dumps(record)+"\\\\n")\\n    assert torch.cuda.device_count() == 1\\n    torch.manual_seed(plan["seed"])\\n    np.random.seed(plan["seed"])\\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\\n    model_dir = plan["model_dir"]\\n    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)\\n    tok.padding_side = "right"\\n    if tok.pad_token_id is None:\\n        tok.pad_token = tok.eos_token\\n    frame = pd.read_parquet("training_frame.parquet")\\n    valid = pd.read_parquet("validation_frame.parquet")\\n    assert set(frame.loc[frame.stage == "host", "group"]).isdisjoint(set(valid.group))\\n    def tokenize(df):\\n        enc = tok(df.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\\n                  add_special_tokens=False)\\n        return [ids+[tok.eos_token_id] for ids in enc["input_ids"]]\\n    tokens, val_tokens = tokenize(frame), tokenize(valid)\\n    def batch(ids, bank):\\n        return {k: v.cuda() for k,v in tok.pad({"input_ids": [bank[i] for i in ids]},\\n                                              padding=True, return_tensors="pt").items()}\\n    model = AttributionModel(model_dir, plan)\\n    model.norm.cuda(); model.head.cuda(); model.projection.cuda()\\n    # Restore only an independently verified external-only checkpoint.\\n    import hashlib\\n    from safetensors.torch import load_file\\n    from peft import set_peft_model_state_dict\\n    checkpoint = Path(plan[\\\'checkpoint\\\'])\\n    for relative,digest in plan[\\\'checkpoint_sha256\\\'].items():\\n        assert hashlib.sha256((checkpoint/relative).read_bytes()).hexdigest()==digest\\n    state = load_file(str(checkpoint/\\\'adapter/adapter_model.safetensors\\\'))\\n    restored = set_peft_model_state_dict(model.encoder,state)\\n    assert not restored.unexpected_keys, restored.unexpected_keys\\n    assert not [k for k in restored.missing_keys if \\\'lora_\\\' in k], restored.missing_keys\\n    heads = torch.load(checkpoint/\\\'heads.pt\\\',map_location=\\\'cpu\\\',weights_only=True)\\n    for name in [\\\'norm\\\',\\\'head\\\',\\\'projection\\\']:\\n        getattr(model,name).load_state_dict(heads[name],strict=True)\\n    assert set(frame.stage)=={\\\'host\\\'}\\n    assert len(frame)==plan[\\\'training_rows\\\'] and len(valid)==plan[\\\'validation_rows\\\']\\n    assert frame.id.is_unique and valid.id.is_unique and set(frame.id).isdisjoint(valid.id)\\n    log(event=\\\'external_checkpoint_loaded\\\',sha256=plan[\\\'checkpoint_sha256\\\'],\\n        fit_rows=len(frame),validation_rows=len(valid))\\n    trainable = [p for p in model.parameters() if p.requires_grad]\\n    # Frozen base is fp16; trainable adapters and heads must be fp32 for GradScaler.\\n    for p in trainable:\\n        p.data = p.data.float()\\n    log(event="loaded", trainable_parameters=sum(p.numel() for p in trainable),\\n        torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,\\n        gpu=torch.cuda.get_device_name(0), training_rows=len(frame), validation_rows=len(valid))\\n    scaler = torch.amp.GradScaler("cuda", init_scale=1024)\\n    skipped = 0\\n    for stage, epochs in [("host", plan["host_epochs"])]:\\n        subset = frame.loc[frame.stage == stage]\\n        micro_steps = math.ceil(len(subset)/plan["batch_size"])\\n        optimizer = torch.optim.AdamW([\\n            {"params": [p for p in model.encoder.parameters() if p.requires_grad], "lr": plan[stage+"_adapter_lr"]},\\n            {"params": list(model.norm.parameters())+list(model.head.parameters())+list(model.projection.parameters()),\\n             "lr": plan[stage+"_head_lr"]}], weight_decay=0.01)\\n        for epoch in range(epochs):\\n            model.train()\\n            optimizer.zero_grad(set_to_none=True)\\n            losses = []\\n            for step, ids in enumerate(paired_batches(subset, micro_steps, plan["seed"] + epoch + (100 if stage == "host" else 0))):\\n                if time.monotonic()-started > plan["wall_limit_seconds"]:\\n                    raise TimeoutError("Frozen pilot wall budget exhausted; no complete result claimed")\\n                labels = torch.as_tensor(frame.loc[ids, "label"].to_numpy(), device="cuda")\\n                with torch.autocast("cuda", dtype=torch.float16):\\n                    logits, features = model(batch(ids, tokens))\\n                    ce = F.cross_entropy(logits.float(), labels)\\n                    contrastive = supcon_loss(features, labels, plan["contrastive_temperature"])\\n                    loss = ce + (plan["contrastive_weight"] * contrastive if variant == "ce_supcon" else 0)\\n                if not torch.isfinite(loss):\\n                    raise FloatingPointError("Nonfinite training loss")\\n                window_start = (step // plan["gradient_accumulation"]) * plan["gradient_accumulation"]\\n                divisor = min(plan["gradient_accumulation"], micro_steps-window_start)\\n                scaler.scale(loss/divisor).backward()\\n                losses.append(float(loss.detach()))\\n                if (step+1) % plan["gradient_accumulation"] == 0 or step+1 == micro_steps:\\n                    scaler.unscale_(optimizer)\\n                    torch.nn.utils.clip_grad_norm_(trainable, 1.0)\\n                    before = scaler.get_scale()\\n                    scaler.step(optimizer)\\n                    scaler.update()\\n                    skipped += int(scaler.get_scale() < before)\\n                    optimizer.zero_grad(set_to_none=True)\\n                if step % 50 == 0:\\n                    log(event="train", stage=stage, epoch=epoch, step=step, steps=micro_steps,\\n                        mean_loss=float(np.mean(losses[-50:])), ce=float(ce.detach()),\\n                        contrastive=float(contrastive.detach()), skipped_steps=skipped,\\n                        peak_gb=torch.cuda.max_memory_allocated()/2**30)\\n            log(event="epoch_complete", stage=stage, epoch=epoch, mean_loss=float(np.mean(losses)))\\n        checkpoint = out / ("external_only" if stage == "pool" else "final")\\n        model.encoder.save_pretrained(checkpoint / "adapter")\\n        torch.save({"norm": model.norm.state_dict(), "head": model.head.state_dict(),\\n                    "projection": model.projection.state_dict()}, checkpoint / "heads.pt")\\n    model.eval()\\n    scores = []\\n    with torch.inference_mode():\\n        for start in range(0, len(valid), 4):\\n            ids = list(range(start, min(start+4, len(valid))))\\n            with torch.autocast("cuda", dtype=torch.float16):\\n                logits, _ = model(batch(ids, val_tokens))\\n            scores.append(logits.float().cpu().numpy())\\n    scores = np.concatenate(scores)\\n    assert np.isfinite(scores).all()\\n    np.savez_compressed(out / "validation.npz", ids=valid.id.to_numpy(dtype=str),\\n                        labels=np.asarray(plan["labels"], dtype=str), logits=scores,\\n                        targets=valid.label.to_numpy())\\n    result = {"variant":variant, "accuracy":float((scores.argmax(1)==valid.label.to_numpy()).mean()),\\n              "seconds":time.monotonic()-started, "skipped_steps":skipped,\\n              "validation_rows":len(valid), "completed":True}\\n    (out / "result.json").write_text(json.dumps(result, indent=2))\\n    log(event="complete", result=result)\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\')\nPath(\'8b_ce_readout.py\').write_text(\'"""Matched frozen readouts for base and v18 adapters; one fixed host fold."""\\nimport gc\\nimport hashlib\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport time\\nimport warnings\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.preprocessing import StandardScaler\\nfrom transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig\\nfrom peft import PeftModel, prepare_model_for_kbit_training\\n\\n\\ndef main():\\n    plan = json.loads(Path("plan.json").read_text())\\n    host = pd.read_csv(plan["host_path"], dtype={"id":str})\\n    upstream = Path(plan["upstream_path"])\\n    original_plan = json.loads((upstream / "plan.json").read_text())\\n    assert original_plan["pilot_fold"] == plan["pilot_fold"]\\n    fitting = pd.read_parquet(upstream / "training_frame.parquet")\\n    validation = pd.read_parquet(upstream / "validation_frame.parquet")\\n    fit_ids = set(fitting.loc[fitting.stage == "host", "id"])\\n    val_ids = set(validation.id)\\n    assert fit_ids.isdisjoint(val_ids) and fit_ids | val_ids == set(host.id)\\n    assert len(fit_ids) == plan["training_rows"] and len(val_ids) == plan["validation_rows"]\\n    train_mask = host.id.isin(fit_ids).to_numpy()\\n    valid_mask = host.id.isin(val_ids).to_numpy()\\n    assert host.loc[valid_mask,"id"].tolist() == validation.id.tolist()\\n    labels = np.asarray(original_plan["labels"])\\n    assert np.array_equal(labels, np.sort(host.model.unique()))\\n    tok = AutoTokenizer.from_pretrained(plan["model_dir"], local_files_only=True)\\n    tok.padding_side = "right"\\n    if tok.pad_token_id is None: tok.pad_token = tok.eos_token\\n    tokenized = tok(host.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\\n                    add_special_tokens=False)["input_ids"]\\n    tokenized = [ids+[tok.eos_token_id] for ids in tokenized]\\n    order = np.argsort([len(ids) for ids in tokenized], kind="stable")\\n    checkpoint_paths = {"ce_external":"ce/external_only/adapter", "ce_final":"ce/final/adapter",\\n                        "supcon_external":"ce_supcon/external_only/adapter", "supcon_final":"ce_supcon/final/adapter"}\\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\\n    assert torch.cuda.device_count() == 1\\n    for variant in json.loads(os.environ["V19_VARIANTS"]):\\n        started = time.monotonic()\\n        print(json.dumps({"variant":variant,"event":"loading"}), flush=True)\\n        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\\n        model = AutoModel.from_pretrained(plan["model_dir"], torch_dtype=torch.float16,\\n            quantization_config=quant, device_map={"":0},\\n            attn_implementation="sdpa", local_files_only=True)\\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)\\n        if variant != "base":\\n            checkpoint = upstream / checkpoint_paths[variant]\\n            weight = checkpoint / "adapter_model.safetensors"\\n            assert hashlib.sha256(weight.read_bytes()).hexdigest() == plan["adapter_sha256"][variant]\\n            model = PeftModel.from_pretrained(model, checkpoint, is_trainable=False)\\n        model = model.eval()\\n        width = model.config.hidden_size\\n        features = np.empty((len(host),len(plan["layers"]),width), dtype=np.float32)\\n        with torch.inference_mode():\\n            for start in range(0,len(host),plan["batch_size"]):\\n                ids = order[start:start+plan["batch_size"]]\\n                enc = {k:v.cuda() for k,v in tok.pad({"input_ids":[tokenized[i] for i in ids]},\\n                                                    padding=True,return_tensors="pt").items()}\\n                with torch.autocast("cuda", dtype=torch.float16):\\n                    states = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\\n                mask = enc["attention_mask"].clone(); mask[:,0] = 0\\n                for j,layer in enumerate(plan["layers"]):\\n                    h = states[layer].float()\\n                    features[ids,j] = ((h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)).cpu().numpy()\\n                del states\\n                if start % 400 == 0:\\n                    print(json.dumps({"variant":variant,"completed":min(start+len(ids),len(host)),\\n                                      "total":len(host),"seconds":time.monotonic()-started}), flush=True)\\n        assert np.isfinite(features).all()\\n        np.save(variant+"_features.npy", features)\\n        del model\\n        gc.collect(); torch.cuda.empty_cache()\\n        predictions = []\\n        diagnostics = {}\\n        for j,layer in enumerate(plan["layers"]):\\n            scaler = StandardScaler()\\n            X = scaler.fit_transform(features[train_mask,j])\\n            V = scaler.transform(features[valid_mask,j])\\n            clf = LogisticRegression(C=plan["C"], max_iter=plan["max_iter"], random_state=plan["seed"])\\n            with warnings.catch_warnings():\\n                warnings.simplefilter("error", ConvergenceWarning)\\n                clf.fit(X,host.loc[train_mask,"model"])\\n            assert np.array_equal(clf.classes_, labels)\\n            scores = clf.predict_log_proba(V)\\n            predictions.append(scores)\\n            diagnostics[str(layer)] = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\\n        scores = np.mean(predictions,axis=0)\\n        accuracy = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\\n        np.savez_compressed(variant+"_validation.npz", ids=host.loc[valid_mask,"id"].to_numpy(dtype=str),\\n                            labels=labels.astype(str), scores=scores, per_layer_scores=np.stack(predictions))\\n        result = {"variant":variant,"accuracy":accuracy,"per_layer_accuracy":diagnostics,\\n                  "seconds":time.monotonic()-started,"fit_rows":int(train_mask.sum()),\\n                  "validation_rows":int(valid_mask.sum()),"feature_shape":list(features.shape),\\n                  "feature_sha256":hashlib.sha256(Path(variant+"_features.npy").read_bytes()).hexdigest()}\\n        Path(variant+"_result.json").write_text(json.dumps(result,indent=2))\\n        print(json.dumps(result),flush=True)\\n        del features\\n        gc.collect()\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\')\nPath(\'4b_small_supcon_worker.py\').write_text(\'"""Standalone GPU worker for the frozen v18 paired-objective pilot."""\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport time\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\n\\ndef supcon_loss(features, labels, temperature=0.1):\\n    """Mean log-probability of same-class positives, excluding self pairs."""\\n    z = F.normalize(features.float(), dim=1)\\n    logits = z @ z.T / temperature\\n    diagonal = torch.eye(len(z), dtype=torch.bool, device=z.device)\\n    positives = labels[:, None].eq(labels[None, :]) & ~diagonal\\n    counts = positives.sum(1)\\n    if not bool((counts > 0).all()):\\n        raise ValueError("Every contrastive anchor must have a distinct positive.")\\n    denominator = torch.logsumexp(logits.masked_fill(diagonal, -torch.inf), dim=1)\\n    return -((logits - denominator[:, None]).masked_fill(~positives, 0).sum(1) / counts).mean()\\n\\n\\ndef paired_batches(frame, steps, seed):\\n    """Two authors, two different prompts each; identical schedule for both arms."""\\n    rng = np.random.default_rng(seed)\\n    groups = {label: {p: sub.index.to_numpy() for p, sub in group.groupby("group")}\\n              for label, group in frame.groupby("label")}\\n    assert all(len(g) >= 2 for g in groups.values())\\n    labels = sorted(groups)\\n    for _ in range(steps):\\n        selected = []\\n        for label in rng.choice(labels, 2, replace=False):\\n            for prompt in rng.choice(list(groups[label]), 2, replace=False):\\n                selected.append(int(rng.choice(groups[label][prompt])))\\n        yield selected\\n\\n\\nclass AttributionModel(nn.Module):\\n    def __init__(self, model_dir, plan):\\n        super().__init__()\\n        from peft import LoraConfig, get_peft_model\\n        from transformers import AutoModel\\n        backbone = AutoModel.from_pretrained(model_dir, torch_dtype=torch.float16,\\n                                            local_files_only=True, attn_implementation="sdpa")\\n        backbone.config.use_cache = False\\n        backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\\n        self.encoder = get_peft_model(backbone, LoraConfig(r=plan["lora_rank"],\\n            lora_alpha=plan["lora_alpha"], target_modules=plan["target_modules"], lora_dropout=0.0,\\n            bias="none"))\\n        width = backbone.config.hidden_size\\n        self.norm = nn.LayerNorm(width)\\n        self.head = nn.Linear(width, 18)\\n        self.projection = nn.Sequential(nn.Linear(width, 256), nn.GELU(), nn.Linear(256, 128))\\n\\n    def forward(self, batch):\\n        hidden = self.encoder(**batch, use_cache=False).last_hidden_state.float()\\n        mask = batch["attention_mask"].clone()\\n        mask[:, 0] = 0\\n        pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)\\n        features = self.norm(pooled)\\n        return self.head(features), self.projection(features)\\n\\n\\ndef main():\\n    import pandas as pd\\n    from transformers import AutoTokenizer\\n    import transformers, peft\\n    plan = json.loads(Path("plan.json").read_text())\\n    variant = os.environ["V18_VARIANT"]\\n    assert variant in plan["variants"]\\n    out = Path(variant)\\n    out.mkdir(exist_ok=True)\\n    started = time.monotonic()\\n    def log(**kw):\\n        record = dict(variant=variant, seconds=round(time.monotonic()-started, 2), **kw)\\n        print(json.dumps(record), flush=True)\\n        with (out / "progress.jsonl").open("a") as f:\\n            f.write(json.dumps(record)+"\\\\n")\\n    assert torch.cuda.device_count() == 1\\n    torch.manual_seed(plan["seed"])\\n    np.random.seed(plan["seed"])\\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\\n    model_dir = plan["model_dir"]\\n    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)\\n    tok.padding_side = "right"\\n    if tok.pad_token_id is None:\\n        tok.pad_token = tok.eos_token\\n    frame = pd.read_parquet("training_frame.parquet")\\n    valid = pd.read_parquet("validation_frame.parquet")\\n    assert set(frame.loc[frame.stage == "host", "group"]).isdisjoint(set(valid.group))\\n    def tokenize(df):\\n        enc = tok(df.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\\n                  add_special_tokens=False)\\n        return [ids+[tok.eos_token_id] for ids in enc["input_ids"]]\\n    tokens, val_tokens = tokenize(frame), tokenize(valid)\\n    def batch(ids, bank):\\n        return {k: v.cuda() for k,v in tok.pad({"input_ids": [bank[i] for i in ids]},\\n                                              padding=True, return_tensors="pt").items()}\\n    model = AttributionModel(model_dir, plan).cuda()\\n    # Restore only an independently verified external-only checkpoint.\\n    import hashlib\\n    from safetensors.torch import load_file\\n    from peft import set_peft_model_state_dict\\n    checkpoint = Path(plan[\\\'checkpoint\\\'])\\n    for relative,digest in plan[\\\'checkpoint_sha256\\\'].items():\\n        assert hashlib.sha256((checkpoint/relative).read_bytes()).hexdigest()==digest\\n    state = load_file(str(checkpoint/\\\'adapter/adapter_model.safetensors\\\'))\\n    restored = set_peft_model_state_dict(model.encoder,state)\\n    assert not restored.unexpected_keys, restored.unexpected_keys\\n    assert not [k for k in restored.missing_keys if \\\'lora_\\\' in k], restored.missing_keys\\n    heads = torch.load(checkpoint/\\\'heads.pt\\\',map_location=\\\'cpu\\\',weights_only=True)\\n    for name in [\\\'norm\\\',\\\'head\\\',\\\'projection\\\']:\\n        getattr(model,name).load_state_dict(heads[name],strict=True)\\n    assert set(frame.stage)=={\\\'host\\\'}\\n    assert len(frame)==plan[\\\'training_rows\\\'] and len(valid)==plan[\\\'validation_rows\\\']\\n    assert frame.id.is_unique and valid.id.is_unique and set(frame.id).isdisjoint(valid.id)\\n    log(event=\\\'external_checkpoint_loaded\\\',sha256=plan[\\\'checkpoint_sha256\\\'],\\n        fit_rows=len(frame),validation_rows=len(valid))\\n    trainable = [p for p in model.parameters() if p.requires_grad]\\n    # Frozen base is fp16; trainable adapters and heads must be fp32 for GradScaler.\\n    for p in trainable:\\n        p.data = p.data.float()\\n    log(event="loaded", trainable_parameters=sum(p.numel() for p in trainable),\\n        torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,\\n        gpu=torch.cuda.get_device_name(0), training_rows=len(frame), validation_rows=len(valid))\\n    scaler = torch.amp.GradScaler("cuda")\\n    skipped = 0\\n    for stage, epochs in [("host", plan["host_epochs"])]:\\n        subset = frame.loc[frame.stage == stage]\\n        micro_steps = math.ceil(len(subset)/plan["batch_size"])\\n        optimizer = torch.optim.AdamW([\\n            {"params": [p for p in model.encoder.parameters() if p.requires_grad], "lr": plan[stage+"_adapter_lr"]},\\n            {"params": list(model.norm.parameters())+list(model.head.parameters())+list(model.projection.parameters()),\\n             "lr": plan[stage+"_head_lr"]}], weight_decay=0.01)\\n        for epoch in range(epochs):\\n            model.train()\\n            optimizer.zero_grad(set_to_none=True)\\n            losses = []\\n            for step, ids in enumerate(paired_batches(subset, micro_steps, plan["seed"] + epoch + (100 if stage == "host" else 0))):\\n                if time.monotonic()-started > plan["wall_limit_seconds"]:\\n                    raise TimeoutError("Frozen pilot wall budget exhausted; no complete result claimed")\\n                labels = torch.as_tensor(frame.loc[ids, "label"].to_numpy(), device="cuda")\\n                with torch.autocast("cuda", dtype=torch.float16):\\n                    logits, features = model(batch(ids, tokens))\\n                    ce = F.cross_entropy(logits.float(), labels)\\n                    contrastive = supcon_loss(features, labels, plan["contrastive_temperature"])\\n                    loss = ce + (plan["contrastive_weight"] * contrastive if variant == "ce_supcon" else 0)\\n                if not torch.isfinite(loss):\\n                    raise FloatingPointError("Nonfinite training loss")\\n                window_start = (step // plan["gradient_accumulation"]) * plan["gradient_accumulation"]\\n                divisor = min(plan["gradient_accumulation"], micro_steps-window_start)\\n                scaler.scale(loss/divisor).backward()\\n                losses.append(float(loss.detach()))\\n                if (step+1) % plan["gradient_accumulation"] == 0 or step+1 == micro_steps:\\n                    scaler.unscale_(optimizer)\\n                    torch.nn.utils.clip_grad_norm_(trainable, 1.0)\\n                    before = scaler.get_scale()\\n                    scaler.step(optimizer)\\n                    scaler.update()\\n                    skipped += int(scaler.get_scale() < before)\\n                    optimizer.zero_grad(set_to_none=True)\\n                if step % 50 == 0:\\n                    log(event="train", stage=stage, epoch=epoch, step=step, steps=micro_steps,\\n                        mean_loss=float(np.mean(losses[-50:])), ce=float(ce.detach()),\\n                        contrastive=float(contrastive.detach()), skipped_steps=skipped,\\n                        peak_gb=torch.cuda.max_memory_allocated()/2**30)\\n            log(event="epoch_complete", stage=stage, epoch=epoch, mean_loss=float(np.mean(losses)))\\n        checkpoint = out / ("external_only" if stage == "pool" else "final")\\n        model.encoder.save_pretrained(checkpoint / "adapter")\\n        torch.save({"norm": model.norm.state_dict(), "head": model.head.state_dict(),\\n                    "projection": model.projection.state_dict()}, checkpoint / "heads.pt")\\n    model.eval()\\n    scores = []\\n    with torch.inference_mode():\\n        for start in range(0, len(valid), 4):\\n            ids = list(range(start, min(start+4, len(valid))))\\n            with torch.autocast("cuda", dtype=torch.float16):\\n                logits, _ = model(batch(ids, val_tokens))\\n            scores.append(logits.float().cpu().numpy())\\n    scores = np.concatenate(scores)\\n    assert np.isfinite(scores).all()\\n    np.savez_compressed(out / "validation.npz", ids=valid.id.to_numpy(dtype=str),\\n                        labels=np.asarray(plan["labels"], dtype=str), logits=scores,\\n                        targets=valid.label.to_numpy())\\n    result = {"variant":variant, "accuracy":float((scores.argmax(1)==valid.label.to_numpy()).mean()),\\n              "seconds":time.monotonic()-started, "skipped_steps":skipped,\\n              "validation_rows":len(valid), "completed":True}\\n    (out / "result.json").write_text(json.dumps(result, indent=2))\\n    log(event="complete", result=result)\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\')\nPath(\'4b_small_supcon_readout.py\').write_text(\'"""Matched frozen readouts for base and v18 adapters; one fixed host fold."""\\nimport gc\\nimport hashlib\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport time\\nimport warnings\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.preprocessing import StandardScaler\\nfrom transformers import AutoModel, AutoTokenizer\\nfrom peft import PeftModel\\n\\n\\ndef main():\\n    plan = json.loads(Path("plan.json").read_text())\\n    host = pd.read_csv(plan["host_path"], dtype={"id":str})\\n    upstream = Path(plan["upstream_path"])\\n    original_plan = json.loads((upstream / "plan.json").read_text())\\n    assert original_plan["pilot_fold"] == plan["pilot_fold"]\\n    fitting = pd.read_parquet(upstream / "training_frame.parquet")\\n    validation = pd.read_parquet(upstream / "validation_frame.parquet")\\n    fit_ids = set(fitting.loc[fitting.stage == "host", "id"])\\n    val_ids = set(validation.id)\\n    assert fit_ids.isdisjoint(val_ids) and fit_ids | val_ids == set(host.id)\\n    assert len(fit_ids) == plan["training_rows"] and len(val_ids) == plan["validation_rows"]\\n    train_mask = host.id.isin(fit_ids).to_numpy()\\n    valid_mask = host.id.isin(val_ids).to_numpy()\\n    assert host.loc[valid_mask,"id"].tolist() == validation.id.tolist()\\n    labels = np.asarray(original_plan["labels"])\\n    assert np.array_equal(labels, np.sort(host.model.unique()))\\n    tok = AutoTokenizer.from_pretrained(plan["model_dir"], local_files_only=True)\\n    tok.padding_side = "right"\\n    if tok.pad_token_id is None: tok.pad_token = tok.eos_token\\n    tokenized = tok(host.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\\n                    add_special_tokens=False)["input_ids"]\\n    tokenized = [ids+[tok.eos_token_id] for ids in tokenized]\\n    order = np.argsort([len(ids) for ids in tokenized], kind="stable")\\n    checkpoint_paths = {"ce_external":"ce/external_only/adapter", "ce_final":"ce/final/adapter",\\n                        "supcon_external":"ce_supcon/external_only/adapter", "supcon_final":"ce_supcon/final/adapter"}\\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\\n    assert torch.cuda.device_count() == 1\\n    for variant in json.loads(os.environ["V19_VARIANTS"]):\\n        started = time.monotonic()\\n        print(json.dumps({"variant":variant,"event":"loading"}), flush=True)\\n        model = AutoModel.from_pretrained(plan["model_dir"], torch_dtype=torch.float16,\\n                                         attn_implementation="sdpa", local_files_only=True)\\n        if variant != "base":\\n            checkpoint = upstream / checkpoint_paths[variant]\\n            weight = checkpoint / "adapter_model.safetensors"\\n            assert hashlib.sha256(weight.read_bytes()).hexdigest() == plan["adapter_sha256"][variant]\\n            model = PeftModel.from_pretrained(model, checkpoint, is_trainable=False)\\n        model = model.cuda().eval()\\n        width = model.config.hidden_size\\n        features = np.empty((len(host),len(plan["layers"]),width), dtype=np.float32)\\n        with torch.inference_mode():\\n            for start in range(0,len(host),plan["batch_size"]):\\n                ids = order[start:start+plan["batch_size"]]\\n                enc = {k:v.cuda() for k,v in tok.pad({"input_ids":[tokenized[i] for i in ids]},\\n                                                    padding=True,return_tensors="pt").items()}\\n                with torch.autocast("cuda", dtype=torch.float16):\\n                    states = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\\n                mask = enc["attention_mask"].clone(); mask[:,0] = 0\\n                for j,layer in enumerate(plan["layers"]):\\n                    h = states[layer].float()\\n                    features[ids,j] = ((h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)).cpu().numpy()\\n                del states\\n                if start % 400 == 0:\\n                    print(json.dumps({"variant":variant,"completed":min(start+len(ids),len(host)),\\n                                      "total":len(host),"seconds":time.monotonic()-started}), flush=True)\\n        assert np.isfinite(features).all()\\n        np.save(variant+"_features.npy", features)\\n        del model\\n        gc.collect(); torch.cuda.empty_cache()\\n        predictions = []\\n        diagnostics = {}\\n        for j,layer in enumerate(plan["layers"]):\\n            scaler = StandardScaler()\\n            X = scaler.fit_transform(features[train_mask,j])\\n            V = scaler.transform(features[valid_mask,j])\\n            clf = LogisticRegression(C=plan["C"], max_iter=plan["max_iter"], random_state=plan["seed"])\\n            with warnings.catch_warnings():\\n                warnings.simplefilter("error", ConvergenceWarning)\\n                clf.fit(X,host.loc[train_mask,"model"])\\n            assert np.array_equal(clf.classes_, labels)\\n            scores = clf.predict_log_proba(V)\\n            predictions.append(scores)\\n            diagnostics[str(layer)] = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\\n        scores = np.mean(predictions,axis=0)\\n        accuracy = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\\n        np.savez_compressed(variant+"_validation.npz", ids=host.loc[valid_mask,"id"].to_numpy(dtype=str),\\n                            labels=labels.astype(str), scores=scores, per_layer_scores=np.stack(predictions))\\n        result = {"variant":variant,"accuracy":accuracy,"per_layer_accuracy":diagnostics,\\n                  "seconds":time.monotonic()-started,"fit_rows":int(train_mask.sum()),\\n                  "validation_rows":int(valid_mask.sum()),"feature_shape":list(features.shape),\\n                  "feature_sha256":hashlib.sha256(Path(variant+"_features.npy").read_bytes()).hexdigest()}\\n        Path(variant+"_result.json").write_text(json.dumps(result,indent=2))\\n        print(json.dumps(result),flush=True)\\n        del features\\n        gc.collect()\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\')\nplan=json.loads("{\\"experiment\\": \\"improvement-v34\\", \\"protocol\\": {\\"experiment\\": \\"improvement-v34\\", \\"kind\\": \\"Multi-fold comparison with the current incumbent component and fixed class balancing\\", \\"status\\": \\"planned_not_launched\\", \\"motivation\\": \\"Saved-prediction diagnostic exposed a difference between pilot and submitted postprocessing. Test two fixed training recipes across more prompts; do not retroactively pass V25.\\", \\"candidate\\": {\\"name\\": \\"8b_ce\\", \\"initialization\\": \\"V25 CE external-only checkpoint, including adapter and heads\\", \\"training_recipe\\": \\"Unchanged V25 CE host stage: two equivalent epochs; fresh optimizer and scaler; same paired sampler and seed\\", \\"readout\\": \\"Unchanged V25 layers 12/24/36 logistic readout, C=0.02, mean log probabilities\\", \\"pilot_scores\\": \\"artifacts/improvement-v25/kaggle-output/ce_final_validation.npz\\"}, \\"comparison\\": {\\"name\\": \\"4b_small_supcon\\", \\"initialization\\": \\"V18 CE+SupCon external-only checkpoint from 4500 selected external rows, including adapter and heads\\", \\"training_recipe\\": \\"Unchanged V18 CE+SupCon host stage: two equivalent epochs; fresh optimizer and scaler; same paired sampler and seed\\", \\"readout\\": \\"Unchanged V19 layers 12/24/36 logistic readout, C=0.02, mean log probabilities\\", \\"pilot_scores\\": \\"artifacts/improvement-v19/kaggle-output/supcon_final_validation.npz\\", \\"reason\\": \\"This recipe supplies V21\'s added component, our current 0.622 public incumbent; V22\'s larger-data component regressed publicly. This is still not validation of the full incumbent ensemble.\\"}, \\"choice_disclosure\\": \\"Candidate CE was chosen after observing V25: it ties SupCon on the balanced pilot blend and has the higher standalone readout. This is post-pilot selection, not a pre-pilot discovery claim.\\", \\"primary_folds\\": {\\"file\\": \\"artifacts/baseline-v1/folds.csv\\", \\"sha256\\": \\"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\\", \\"reuse\\": [0], \\"train\\": [1, 2, 3, 4]}, \\"related_confirmation\\": {\\"file\\": \\"artifacts/improvement-v4/robustness/folds.csv\\", \\"sha256\\": \\"090c42e940fcfd380d821e2e7a6414605e301f826e5e6e29ad678be5643eee32\\", \\"fold\\": 0, \\"launch_requires\\": \\"All primary gates pass\\"}, \\"scoring\\": {\\"sparse_reference\\": \\"V32 fixed V17 sparse OOF; related holdout fits the identical specification using fixed V17 external IDs and train-only vocabulary\\", \\"component_weight\\": 0.15, \\"blend\\": \\"sparse decision scores + 0.15 * mean neural log probabilities\\", \\"postprocessing\\": \\"Softmax then credited V16 equal-class-mass balancing for exactly 300 iterations, separately on each held-out fold; no prompt-wise one-to-one assignment\\", \\"metric\\": \\"Balanced accuracy (mean recall across 18 source labels)\\", \\"no_selection\\": \\"No weight, epoch, layer, temperature, seed or objective search. Evaluate fixed final checkpoints. Report ordinary accuracy as a diagnostic only.\\"}, \\"primary_gate\\": {\\"candidate_pooled_balanced_accuracy_minimum\\": 0.52, \\"candidate_minus_comparison_minimum\\": 0.02, \\"materially_positive_gain_folds_minimum\\": 4, \\"material_tolerance\\": 1e-08, \\"paired_prompt_bootstrap_gain_lower_must_be_positive\\": true}, \\"related_gate\\": {\\"candidate_balanced_accuracy_minimum\\": 0.5, \\"candidate_minus_comparison_minimum\\": 0.015}, \\"bootstrap\\": {\\"samples\\": 10000, \\"seed\\": 3401, \\"unit\\": \\"Whole prompt groups, stratified by primary fold; recompute pooled balanced accuracy for each paired resample\\", \\"interval\\": [0.025, 0.975]}, \\"runtime\\": {\\"evidence\\": \\"V25 CE host stage through completion about 3811 seconds plus 557-second readout; V18 SupCon about 1483 seconds plus 264-second V19 readout\\", \\"execution\\": \\"One GPU per recipe, four primary folds sequentially within each recipe; related fold only after primary qualification\\", \\"primary_projection_seconds\\": 18392, \\"wall_limit_seconds\\": 24300, \\"launch_requires\\": \\"Independent verification of external-only artifacts, exact train/holdout boundaries, notebook source, offline model dependencies and sufficient current Kaggle quota\\", \\"paid_compute\\": false}, \\"attribution\\": \\"Public pool and fixed balancing: Nishant Dahal and original contributors, CC BY-SA 4.0. Qwen backbones: Qwen. LoRA, supervised contrastive learning and QLoRA retain the cited V18/V25 method credits. This is a new evaluation protocol, not a newly invented training method.\\", \\"limitations\\": [\\"Full credited 14B/DeBERTa ensemble is not included in local OOF; only an actual submission can prove a gain over 0.622\\", \\"Pilot fold zero is reused; all development labels have been inspected before\\", \\"Size, quantization, adapter coverage, objective and external-pool size differ between recipes; no pure scaling or causal method claim\\", \\"Equal hidden class mass is an assumption; balancing is fixed because the submitted baseline already uses it\\", \\"Related confirmation is one predeclared holdout, not full related-fold OOF\\", \\"Passing supports separate full-host training and inference validation; a fold model is never submitted directly\\", \\"V25 and V30 remain failed under their original gates; V33 remains unlaunched\\"]}, \\"protocol_sha256\\": \\"c5fde950434c1dd604a3233d9b989790d9af105de9c3d732172c73f3539a35db\\", \\"host_sha256\\": \\"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\\", \\"labels\\": [\\"claude-fable-5\\", \\"claude-opus-4-6\\", \\"claude-opus-4-8\\", \\"claude-opus-5\\", \\"claude-sonnet-5\\", \\"gemini-3.1-pro-preview\\", \\"gemini-3.6-flash\\", \\"gemini-3.7-flash\\", \\"gemini-3.8-flash\\", \\"glm-5.3\\", \\"glm-5.3-flash\\", \\"gpt-5.5\\", \\"gpt-5.6-luna\\", \\"gpt-5.6-sol\\", \\"gpt-5.6-terra\\", \\"gpt-6-astra\\", \\"kimi-k3\\", \\"qwen3.8-max\\"], \\"fold_by_prompt\\": {\\"p000\\": {\\"group\\": \\"p000\\", \\"fold\\": 3}, \\"p001\\": {\\"group\\": \\"p001\\", \\"fold\\": 3}, \\"p002\\": {\\"group\\": \\"p002\\", \\"fold\\": 4}, \\"p003\\": {\\"group\\": \\"p003\\", \\"fold\\": 3}, \\"p004\\": {\\"group\\": \\"p004\\", \\"fold\\": 2}, \\"p005\\": {\\"group\\": \\"p005\\", \\"fold\\": 1}, \\"p006\\": {\\"group\\": \\"p006\\", \\"fold\\": 4}, \\"p007\\": {\\"group\\": \\"p007\\", \\"fold\\": 4}, \\"p008\\": {\\"group\\": \\"p008\\", \\"fold\\": 3}, \\"p009\\": {\\"group\\": \\"p009\\", \\"fold\\": 0}, \\"p010\\": {\\"group\\": \\"p010\\", \\"fold\\": 4}, \\"p011\\": {\\"group\\": \\"p011\\", \\"fold\\": 2}, \\"p012\\": {\\"group\\": \\"p012\\", \\"fold\\": 3}, \\"p013\\": {\\"group\\": \\"p013\\", \\"fold\\": 4}, \\"p014\\": {\\"group\\": \\"p014\\", \\"fold\\": 4}, \\"p015\\": {\\"group\\": \\"p015\\", \\"fold\\": 3}, \\"p016\\": {\\"group\\": \\"p016\\", \\"fold\\": 3}, \\"p017\\": {\\"group\\": \\"p017\\", \\"fold\\": 4}, \\"p018\\": {\\"group\\": \\"p018\\", \\"fold\\": 2}, \\"p019\\": {\\"group\\": \\"p019\\", \\"fold\\": 0}, \\"p020\\": {\\"group\\": \\"p020\\", \\"fold\\": 3}, \\"p021\\": {\\"group\\": \\"p021\\", \\"fold\\": 2}, \\"p022\\": {\\"group\\": \\"p022\\", \\"fold\\": 0}, \\"p023\\": {\\"group\\": \\"p023\\", \\"fold\\": 2}, \\"p024\\": {\\"group\\": \\"p024\\", \\"fold\\": 2}, \\"p025\\": {\\"group\\": \\"p025\\", \\"fold\\": 3}, \\"p026\\": {\\"group\\": \\"p026\\", \\"fold\\": 3}, \\"p027\\": {\\"group\\": \\"p027\\", \\"fold\\": 0}, \\"p028\\": {\\"group\\": \\"p028\\", \\"fold\\": 2}, \\"p029\\": {\\"group\\": \\"p029\\", \\"fold\\": 2}, \\"p030\\": {\\"group\\": \\"p030\\", \\"fold\\": 4}, \\"p031\\": {\\"group\\": \\"p031\\", \\"fold\\": 2}, \\"p032\\": {\\"group\\": \\"p032\\", \\"fold\\": 0}, \\"p033\\": {\\"group\\": \\"p033\\", \\"fold\\": 1}, \\"p034\\": {\\"group\\": \\"p034\\", \\"fold\\": 4}, \\"p035\\": {\\"group\\": \\"p035\\", \\"fold\\": 2}, \\"p036\\": {\\"group\\": \\"p036\\", \\"fold\\": 0}, \\"p037\\": {\\"group\\": \\"p037\\", \\"fold\\": 1}, \\"p038\\": {\\"group\\": \\"p038\\", \\"fold\\": 3}, \\"p039\\": {\\"group\\": \\"p039\\", \\"fold\\": 2}, \\"p040\\": {\\"group\\": \\"p040\\", \\"fold\\": 1}, \\"p041\\": {\\"group\\": \\"p041\\", \\"fold\\": 1}, \\"p042\\": {\\"group\\": \\"p042\\", \\"fold\\": 0}, \\"p043\\": {\\"group\\": \\"p043\\", \\"fold\\": 2}, \\"p044\\": {\\"group\\": \\"p044\\", \\"fold\\": 2}, \\"p045\\": {\\"group\\": \\"p045\\", \\"fold\\": 2}, \\"p046\\": {\\"group\\": \\"p046\\", \\"fold\\": 3}, \\"p047\\": {\\"group\\": \\"p047\\", \\"fold\\": 1}, \\"p048\\": {\\"group\\": \\"p048\\", \\"fold\\": 2}, \\"p049\\": {\\"group\\": \\"p049\\", \\"fold\\": 1}, \\"p050\\": {\\"group\\": \\"p050\\", \\"fold\\": 4}, \\"p051\\": {\\"group\\": \\"p051\\", \\"fold\\": 4}, \\"p052\\": {\\"group\\": \\"p052\\", \\"fold\\": 1}, \\"p053\\": {\\"group\\": \\"p053\\", \\"fold\\": 2}, \\"p054\\": {\\"group\\": \\"p054\\", \\"fold\\": 1}, \\"p055\\": {\\"group\\": \\"p055\\", \\"fold\\": 3}, \\"p056\\": {\\"group\\": \\"p056\\", \\"fold\\": 1}, \\"p057\\": {\\"group\\": \\"p057\\", \\"fold\\": 0}, \\"p058\\": {\\"group\\": \\"p055\\", \\"fold\\": 3}, \\"p059\\": {\\"group\\": \\"p059\\", \\"fold\\": 4}, \\"p060\\": {\\"group\\": \\"p060\\", \\"fold\\": 3}, \\"p061\\": {\\"group\\": \\"p061\\", \\"fold\\": 4}, \\"p062\\": {\\"group\\": \\"p062\\", \\"fold\\": 0}, \\"p063\\": {\\"group\\": \\"p063\\", \\"fold\\": 4}, \\"p064\\": {\\"group\\": \\"p064\\", \\"fold\\": 1}, \\"p065\\": {\\"group\\": \\"p065\\", \\"fold\\": 0}, \\"p066\\": {\\"group\\": \\"p066\\", \\"fold\\": 1}, \\"p067\\": {\\"group\\": \\"p067\\", \\"fold\\": 3}, \\"p068\\": {\\"group\\": \\"p068\\", \\"fold\\": 0}, \\"p069\\": {\\"group\\": \\"p069\\", \\"fold\\": 4}, \\"p070\\": {\\"group\\": \\"p070\\", \\"fold\\": 0}, \\"p071\\": {\\"group\\": \\"p071\\", \\"fold\\": 3}, \\"p072\\": {\\"group\\": \\"p072\\", \\"fold\\": 1}, \\"p073\\": {\\"group\\": \\"p073\\", \\"fold\\": 1}, \\"p074\\": {\\"group\\": \\"p074\\", \\"fold\\": 3}, \\"p075\\": {\\"group\\": \\"p075\\", \\"fold\\": 1}, \\"p076\\": {\\"group\\": \\"p076\\", \\"fold\\": 2}, \\"p077\\": {\\"group\\": \\"p077\\", \\"fold\\": 4}, \\"p078\\": {\\"group\\": \\"p078\\", \\"fold\\": 0}, \\"p079\\": {\\"group\\": \\"p079\\", \\"fold\\": 0}, \\"p080\\": {\\"group\\": \\"p080\\", \\"fold\\": 1}, \\"p081\\": {\\"group\\": \\"p081\\", \\"fold\\": 2}, \\"p082\\": {\\"group\\": \\"p010\\", \\"fold\\": 4}, \\"p083\\": {\\"group\\": \\"p083\\", \\"fold\\": 1}, \\"p084\\": {\\"group\\": \\"p084\\", \\"fold\\": 2}, \\"p085\\": {\\"group\\": \\"p085\\", \\"fold\\": 0}, \\"p086\\": {\\"group\\": \\"p086\\", \\"fold\\": 0}, \\"p087\\": {\\"group\\": \\"p087\\", \\"fold\\": 4}, \\"p088\\": {\\"group\\": \\"p088\\", \\"fold\\": 1}, \\"p089\\": {\\"group\\": \\"p089\\", \\"fold\\": 0}, \\"p090\\": {\\"group\\": \\"p090\\", \\"fold\\": 1}, \\"p091\\": {\\"group\\": \\"p091\\", \\"fold\\": 4}, \\"p092\\": {\\"group\\": \\"p092\\", \\"fold\\": 1}, \\"p093\\": {\\"group\\": \\"p093\\", \\"fold\\": 1}, \\"p094\\": {\\"group\\": \\"p094\\", \\"fold\\": 1}, \\"p095\\": {\\"group\\": \\"p095\\", \\"fold\\": 3}, \\"p096\\": {\\"group\\": \\"p096\\", \\"fold\\": 2}, \\"p097\\": {\\"group\\": \\"p097\\", \\"fold\\": 3}, \\"p098\\": {\\"group\\": \\"p098\\", \\"fold\\": 2}, \\"p099\\": {\\"group\\": \\"p099\\", \\"fold\\": 4}, \\"p100\\": {\\"group\\": \\"p100\\", \\"fold\\": 0}, \\"p101\\": {\\"group\\": \\"p101\\", \\"fold\\": 0}, \\"p102\\": {\\"group\\": \\"p102\\", \\"fold\\": 2}, \\"p103\\": {\\"group\\": \\"p103\\", \\"fold\\": 4}, \\"p104\\": {\\"group\\": \\"p104\\", \\"fold\\": 4}, \\"p105\\": {\\"group\\": \\"p105\\", \\"fold\\": 3}, \\"p106\\": {\\"group\\": \\"p106\\", \\"fold\\": 3}, \\"p107\\": {\\"group\\": \\"p107\\", \\"fold\\": 3}, \\"p108\\": {\\"group\\": \\"p108\\", \\"fold\\": 0}, \\"p109\\": {\\"group\\": \\"p109\\", \\"fold\\": 3}, \\"p110\\": {\\"group\\": \\"p110\\", \\"fold\\": 0}, \\"p111\\": {\\"group\\": \\"p004\\", \\"fold\\": 2}, \\"p112\\": {\\"group\\": \\"p112\\", \\"fold\\": 1}, \\"p113\\": {\\"group\\": \\"p113\\", \\"fold\\": 4}, \\"p114\\": {\\"group\\": \\"p114\\", \\"fold\\": 1}, \\"p115\\": {\\"group\\": \\"p115\\", \\"fold\\": 4}, \\"p116\\": {\\"group\\": \\"p116\\", \\"fold\\": 2}, \\"p117\\": {\\"group\\": \\"p117\\", \\"fold\\": 2}, \\"p118\\": {\\"group\\": \\"p118\\", \\"fold\\": 3}, \\"p119\\": {\\"group\\": \\"p119\\", \\"fold\\": 0}, \\"p120\\": {\\"group\\": \\"p120\\", \\"fold\\": 4}, \\"p121\\": {\\"group\\": \\"p121\\", \\"fold\\": 4}, \\"p122\\": {\\"group\\": \\"p122\\", \\"fold\\": 0}, \\"p123\\": {\\"group\\": \\"p123\\", \\"fold\\": 2}, \\"p124\\": {\\"group\\": \\"p124\\", \\"fold\\": 1}, \\"p125\\": {\\"group\\": \\"p125\\", \\"fold\\": 1}, \\"p126\\": {\\"group\\": \\"p126\\", \\"fold\\": 3}, \\"p127\\": {\\"group\\": \\"p127\\", \\"fold\\": 3}, \\"p128\\": {\\"group\\": \\"p128\\", \\"fold\\": 3}, \\"p129\\": {\\"group\\": \\"p129\\", \\"fold\\": 0}, \\"p130\\": {\\"group\\": \\"p130\\", \\"fold\\": 0}, \\"p131\\": {\\"group\\": \\"p008\\", \\"fold\\": 3}, \\"p132\\": {\\"group\\": \\"p132\\", \\"fold\\": 1}, \\"p133\\": {\\"group\\": \\"p133\\", \\"fold\\": 2}, \\"p134\\": {\\"group\\": \\"p134\\", \\"fold\\": 0}, \\"p135\\": {\\"group\\": \\"p135\\", \\"fold\\": 2}, \\"p136\\": {\\"group\\": \\"p128\\", \\"fold\\": 3}, \\"p137\\": {\\"group\\": \\"p137\\", \\"fold\\": 1}, \\"p138\\": {\\"group\\": \\"p138\\", \\"fold\\": 4}, \\"p139\\": {\\"group\\": \\"p139\\", \\"fold\\": 4}, \\"p140\\": {\\"group\\": \\"p140\\", \\"fold\\": 3}, \\"p141\\": {\\"group\\": \\"p141\\", \\"fold\\": 0}, \\"p142\\": {\\"group\\": \\"p142\\", \\"fold\\": 2}, \\"p143\\": {\\"group\\": \\"p143\\", \\"fold\\": 0}, \\"p144\\": {\\"group\\": \\"p144\\", \\"fold\\": 4}, \\"p145\\": {\\"group\\": \\"p145\\", \\"fold\\": 0}, \\"p146\\": {\\"group\\": \\"p100\\", \\"fold\\": 0}, \\"p147\\": {\\"group\\": \\"p147\\", \\"fold\\": 1}, \\"p148\\": {\\"group\\": \\"p017\\", \\"fold\\": 4}, \\"p149\\": {\\"group\\": \\"p149\\", \\"fold\\": 2}}, \\"fold_sizes\\": {\\"0\\": {\\"training_rows\\": 2159, \\"validation_rows\\": 540}}, \\"arms\\": {\\"8b_ce\\": {\\"parameters\\": {\\"experiment\\": \\"improvement-v25\\", \\"kind\\": \\"8B NF4 all-linear paired CE/SupCon pilot\\", \\"model\\": \\"qwen-lm/qwen-3/Transformers/8b-base/1\\", \\"seed\\": 1801, \\"pilot_fold\\": 0, \\"external_cap_per_class\\": 1000, \\"external_host_cosine_exclusion\\": 0.3, \\"max_length\\": 256, \\"batch_size\\": 4, \\"gradient_accumulation\\": 8, \\"lora_rank\\": 16, \\"lora_alpha\\": 32, \\"target_modules\\": \\"all-linear\\", \\"pool_epochs\\": 1, \\"host_epochs\\": 2, \\"pool_adapter_lr\\": 0.0001, \\"pool_head_lr\\": 0.0005, \\"host_adapter_lr\\": 3e-05, \\"host_head_lr\\": 0.0001, \\"contrastive_weight\\": 0.1, \\"contrastive_temperature\\": 0.1, \\"variants\\": [\\"ce\\", \\"ce_supcon\\"], \\"checkpoint_rule\\": \\"Evaluate fixed final host epoch only; no best-epoch selection\\", \\"promotion\\": \\"Fixed three-layer readout must reach >=0.4625925926 and fixed 0.15 sparse-pilot blend >=0.5311111111. Select highest blend among qualified arms, report two-arm selection. No direct pilot submission.\\", \\"wall_limit_seconds\\": 24300, \\"sources\\": [\\"https://arxiv.org/html/2004.11362v5\\", \\"https://github.com/HobbitLong/SupContrast\\", \\"https://arxiv.org/html/2106.09685v2\\", \\"https://github.com/microsoft/LoRA\\", \\"https://huggingface.co/docs/peft/package_reference/lora\\", \\"https://arxiv.org/abs/2305.14314\\", \\"https://github.com/artidoro/qlora\\", \\"https://huggingface.co/docs/peft/developer_guides/quantization\\", \\"https://huggingface.co/Qwen/Qwen3-8B-Base/raw/main/config.json\\"], \\"attribution\\": \\"LoRA and supervised contrastive learning are established methods. Joint CE plus cross-prompt supervised contrastive adaptation here is our experimental combination, not a reproduction of the image-learning paper. Public pool credited to Nishant Dahal and original contributors (CC BY-SA 4.0). QLoRA: Dettmers et al. 2305.14314; implementation uses Hugging Face PEFT/bitsandbytes. Uses ordinary AdamW and fp16 compute on T4.\\", \\"limitations\\": [\\"Same repeatedly inspected 540-row fold; not an untouched test\\", \\"Changes model size, quantization and adapter coverage together; no pure scaling attribution\\", \\"Two objectives compared; no full-ensemble CV claim\\", \\"QLoRA instruction-tuning results do not predict attribution accuracy\\", \\"No paid compute\\"], \\"comparison\\": {\\"v22_supcon_readout\\": 0.4425925926, \\"v22_supcon_sparse_blend\\": 0.5111111111}, \\"readout\\": {\\"layers\\": [12, 24, 36], \\"C\\": 0.02, \\"max_iter\\": 1000, \\"seed\\": 1901, \\"batch_size\\": 8}, \\"differences_from_v18\\": [\\"18000 external rows as V22\\", \\"8B NF4 base with all-linear rank16 adapters instead of 4B fp16 q/v-only\\", \\"Unchanged fixed readout layers12/24/36 and C0.02\\"], \\"reason\\": \\"Test a stronger trainable reader, with matched CE and SupCon objectives; require two-point gains over V22 selected component and sparse blend.\\", \\"launch_gate\\": \\"V24 completed finite 8B NF4 updates and adapter reload; projected 5580 training microsteps plus conservative 1800s readout/loading allowance <=24300s (25% below 9h).\\", \\"quantization\\": {\\"load_in_4bit\\": true, \\"quant_type\\": \\"nf4\\", \\"double_quant\\": true, \\"compute_dtype\\": \\"float16\\"}, \\"host_sha256\\": \\"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\\", \\"pool_sha256\\": \\"37750e899ff6634c147daa7888bfed048e92d3fb1381fe5d21f93bf72a84e8ec\\", \\"folds_sha256\\": \\"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\\", \\"labels\\": [\\"claude-fable-5\\", \\"claude-opus-4-6\\", \\"claude-opus-4-8\\", \\"claude-opus-5\\", \\"claude-sonnet-5\\", \\"gemini-3.1-pro-preview\\", \\"gemini-3.6-flash\\", \\"gemini-3.7-flash\\", \\"gemini-3.8-flash\\", \\"glm-5.3\\", \\"glm-5.3-flash\\", \\"gpt-5.5\\", \\"gpt-5.6-luna\\", \\"gpt-5.6-sol\\", \\"gpt-5.6-terra\\", \\"gpt-6-astra\\", \\"kimi-k3\\", \\"qwen3.8-max\\"]}, \\"variant\\": \\"ce\\", \\"readout_variant\\": \\"ce_final\\", \\"training_worker\\": \\"8b_ce_worker.py\\", \\"readout_worker\\": \\"8b_ce_readout.py\\", \\"readout\\": {\\"layers\\": [12, 24, 36], \\"C\\": 0.02, \\"max_iter\\": 1000, \\"seed\\": 1901, \\"batch_size\\": 8}, \\"checkpoint_sha256\\": {\\"heads.pt\\": \\"8994fda7fdb22667eece5a84766238e08d3710a2fa23e9bb9c43c75fd876d205\\", \\"adapter/adapter_model.safetensors\\": \\"63fd0bbdf7dcdea5b938ef7536016db23e1edc8ed1f15c5f1d21b682ffd1736e\\", \\"adapter/README.md\\": \\"43064eed5a371c72e6afd650a4d8f98837b6026ba9068fbf62ef1887fb70cf55\\", \\"adapter/adapter_config.json\\": \\"abb8632c980385f5503d1c4acbd9cf4bd4838277967a0d9115d178922d860b61\\"}, \\"hidden_size\\": 4096, \\"origin_version\\": 25, \\"origin_worker_sha256\\": \\"12c5b7a5f7fb1384dbbd864134b281fe8d3699b61682a598c79aa2b62b722f6d\\", \\"origin_readout_sha256\\": \\"9f6ba4e1bbf6cd0556a53b625819f280609044f8e84579da36dcf31bda95dc66\\"}, \\"4b_small_supcon\\": {\\"parameters\\": {\\"experiment\\": \\"improvement-v18\\", \\"kind\\": \\"paired training-objective pilot\\", \\"model\\": \\"qwen-lm/qwen-3/Transformers/4b-base/1\\", \\"seed\\": 1801, \\"pilot_fold\\": 0, \\"external_cap_per_class\\": 250, \\"external_host_cosine_exclusion\\": 0.3, \\"max_length\\": 256, \\"batch_size\\": 4, \\"gradient_accumulation\\": 8, \\"lora_rank\\": 16, \\"lora_alpha\\": 32, \\"target_modules\\": [\\"q_proj\\", \\"v_proj\\"], \\"pool_epochs\\": 1, \\"host_epochs\\": 2, \\"pool_adapter_lr\\": 0.0001, \\"pool_head_lr\\": 0.0005, \\"host_adapter_lr\\": 3e-05, \\"host_head_lr\\": 0.0001, \\"contrastive_weight\\": 0.1, \\"contrastive_temperature\\": 0.1, \\"variants\\": [\\"ce\\", \\"ce_supcon\\"], \\"checkpoint_rule\\": \\"Evaluate fixed final host epoch only; no best-epoch selection\\", \\"promotion\\": \\"SupCon must improve final pilot accuracy by at least 0.01 over identical CE training before broader confirmation. Neither pilot alone qualifies for submission.\\", \\"wall_limit_seconds\\": 24300, \\"sources\\": [\\"https://arxiv.org/html/2004.11362v5\\", \\"https://github.com/HobbitLong/SupContrast\\", \\"https://arxiv.org/html/2106.09685v2\\", \\"https://github.com/microsoft/LoRA\\", \\"https://huggingface.co/docs/peft/package_reference/lora\\"], \\"attribution\\": \\"LoRA and supervised contrastive learning are established methods. Joint CE plus cross-prompt supervised contrastive adaptation here is our experimental combination, not a reproduction of the image-learning paper. Public pool credited to Nishant Dahal and original contributors (CC BY-SA 4.0).\\", \\"limitations\\": [\\"One repeatedly inspected host fold, not untouched confirmation\\", \\"4500-row external cap screens objectives before scaling\\", \\"Small contrastive batches may limit the benefit\\", \\"Host-based external near-match exclusion uses unlabeled host text only for conservative leakage screening\\"], \\"host_sha256\\": \\"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\\", \\"pool_sha256\\": \\"37750e899ff6634c147daa7888bfed048e92d3fb1381fe5d21f93bf72a84e8ec\\", \\"folds_sha256\\": \\"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\\", \\"labels\\": [\\"claude-fable-5\\", \\"claude-opus-4-6\\", \\"claude-opus-4-8\\", \\"claude-opus-5\\", \\"claude-sonnet-5\\", \\"gemini-3.1-pro-preview\\", \\"gemini-3.6-flash\\", \\"gemini-3.7-flash\\", \\"gemini-3.8-flash\\", \\"glm-5.3\\", \\"glm-5.3-flash\\", \\"gpt-5.5\\", \\"gpt-5.6-luna\\", \\"gpt-5.6-sol\\", \\"gpt-5.6-terra\\", \\"gpt-6-astra\\", \\"kimi-k3\\", \\"qwen3.8-max\\"]}, \\"variant\\": \\"ce_supcon\\", \\"readout_variant\\": \\"supcon_final\\", \\"training_worker\\": \\"4b_small_supcon_worker.py\\", \\"readout_worker\\": \\"4b_small_supcon_readout.py\\", \\"readout\\": {\\"layers\\": [12, 24, 36], \\"C\\": 0.02, \\"max_iter\\": 1000, \\"seed\\": 1901, \\"batch_size\\": 8}, \\"checkpoint_sha256\\": {\\"heads.pt\\": \\"6c540f09084be67029f3651a020214fcb2afe53e95a127486985bf521a7e10ff\\", \\"adapter/adapter_model.safetensors\\": \\"353f3e3bacab0660d60b8b8e4a90e4d258f26fa5047a4010db3f2227b7bfb6d5\\", \\"adapter/README.md\\": \\"c7094bd33e036ce53b5d373d4c251d03e96d7b624f3ce89c9805e5f5844dd79b\\", \\"adapter/adapter_config.json\\": \\"5cc84f15952fcb3d9c192cc8d819fc201a31c73f95884656c5fe2acabda33ab7\\"}, \\"hidden_size\\": 2560, \\"origin_version\\": 18, \\"origin_worker_sha256\\": \\"a79591118969d7e2b687523a64c46f68399a217c36972d37e300aaba7d440b34\\", \\"origin_readout_sha256\\": \\"c55bed7580eefde8f7461fff084510cd2c83fd5e0768cff066ae1ff700a4409c\\"}}, \\"source_sha256\\": {\\"run_confirmation_arm_v34.py\\": \\"91a03e111eda97d4e7c9e0203c87241487ceb39c7d7e18027c80cb0793a3fe91\\", \\"prepare_likelihood_v33.py\\": \\"9517acff96c5f73c2838a3cc06bdc20aadeff9e14c801b460e3d6cd28c222838\\", \\"8b_ce_worker.py\\": \\"33c117d136081b7a330becdcd8232bef58eae7b5611c4d807fffb6cf9ea65c82\\", \\"8b_ce_readout.py\\": \\"fa1fe90c97657ac6b06175192244dfb53ecb7dc18f946975cb722f1caaa33662\\", \\"4b_small_supcon_worker.py\\": \\"6d3371ddfef1c49d2a364c5e0b0f9152095c39d3c5302e2f8bdeb262ab16d96c\\", \\"4b_small_supcon_readout.py\\": \\"60ca39dfd35b1be99d73c0671776eee57feb193b480c20ba101c7d7761770fbb\\"}, \\"primary_audit_sha256\\": \\"c08145212c44ab6b3b6e0064db5178ece7a01ec6005477ec97c356f467352ead\\", \\"primary_oof_sha256\\": \\"6609dcecf4cd18bc5fbb901794feb90a7e2e9f78485f572942820b30258c4021\\", \\"confirmation_kind\\": \\"related\\", \\"confirmation_folds\\": [0]}")\n\nimport torch\nassert torch.cuda.device_count()==2\nhosts=list(Path(\'/kaggle/input\').rglob(\'train.csv\'))\nassert len(hosts)==1\nassert hashlib.sha256(hosts[0].read_bytes()).hexdigest()==plan[\'host_sha256\']\nplan[\'host_path\']=str(hosts[0])\nfor arm,spec in plan[\'arms\'].items():\n    candidates=[]\n    for p in Path(\'/kaggle/input\').rglob(\'config.json\'):\n        cfg=json.loads(p.read_text())\n        if cfg.get(\'model_type\')==\'qwen3\' and cfg.get(\'hidden_size\')==spec[\'hidden_size\'] and cfg.get(\'num_hidden_layers\')==36:\n            candidates.append(p.parent)\n    assert len(candidates)==1,(arm,candidates)\n    spec[\'model_dir\']=str(candidates[0])\n    weights=[p for p in Path(\'/kaggle/input\').rglob(\'adapter_model.safetensors\')\n             if p.parent.parent.name==\'external_only\' and p.parent.parent.parent.name==spec[\'variant\']\n             and hashlib.sha256(p.read_bytes()).hexdigest()==spec[\'checkpoint_sha256\'][\'adapter/adapter_model.safetensors\']]\n    assert len(weights)==1,(arm,len(weights))\n    spec[\'checkpoint\']=str(weights[0].parent.parent)\n    for relative,digest in spec[\'checkpoint_sha256\'].items():\n        assert hashlib.sha256((Path(spec[\'checkpoint\'])/relative).read_bytes()).hexdigest()==digest\nplan[\'deadline_unix\']=time.time()+plan[\'protocol\'][\'runtime\'][\'wall_limit_seconds\']\nPath(\'plan.json\').write_text(json.dumps(plan,indent=2))\nprocesses=[]\ntry:\n    for gpu,arm in enumerate(plan[\'arms\']):\n        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu))\n        processes.append(subprocess.Popen([sys.executable,\'-u\',\'run_confirmation_arm_v34.py\',arm],env=env,start_new_session=True))\n    while any(p.poll() is None for p in processes):\n        if any(p.poll() not in (None,0) for p in processes):raise RuntimeError(\'Confirmation arm failed\')\n        if time.time()>plan[\'deadline_unix\']:raise TimeoutError(\'Confirmation notebook wall budget exhausted\')\n        time.sleep(10)\n    assert all(p.returncode==0 for p in processes)\nexcept BaseException:\n    for p in processes:\n        if p.poll() is None:os.killpg(p.pid,signal.SIGTERM)\n    for p in processes:p.wait()\n    raise\nsummary={\'completed\':True,\'experiment\':\'improvement-v34\',\'submission_authorized\':False,\n         \'decision\':\'Independent related-holdout audit and frozen related gates required\'}\nPath(\'summary.json\').write_text(json.dumps(summary,indent=2))\nprint(json.dumps(summary),flush=True)\n'
assert related_code.count("Path('/kaggle/input')")==3
related_code=related_code.replace("Path('/kaggle/input')",f"Path({str(V37_TRAINING_INPUT)!r})")
related_output=V37_ROOT/'artifacts/improvement-v34/related/kaggle-output'
related_output.mkdir(exist_ok=False)
Path('v37-related-run.py').write_text(related_code)
run_stage([sys.executable,'-u',str(V37_ROOT/'v37-related-run.py')],cwd=related_output,check=True,timeout=min(9000,remaining_budget()))
run_stage([sys.executable,'-u','scripts/evaluate_related_v34.py'],check=True,timeout=min(1800,remaining_budget()))
V37_GATE=json.loads(Path('artifacts/improvement-v34/related/evaluation-audit.json').read_text())
assert V37_GATE['verified']
V37_GATE_PASSED=bool(V37_GATE['qualifies_for_full_host_training'])
if V37_GATE_PASSED:
    run_stage([sys.executable,'-u','scripts/build_full_v34.py'],check=True,timeout=min(120,remaining_budget()))
    V37_FULL_ROOT=V37_ROOT/'artifacts/improvement-v34/full-host/kaggle-output'
    V37_FULL_ROOT.mkdir(exist_ok=False)
    env=dict(os.environ,V34_INPUT_ROOT=str(V37_TRAINING_INPUT))
    run_stage([sys.executable,'-u',str(V37_ROOT/'artifacts/improvement-v34/full-host/run.py')],cwd=V37_FULL_ROOT,env=env,check=True,timeout=min(9300,remaining_budget()))
    run_stage([sys.executable,'-u','scripts/audit_full_v34.py'],check=True,timeout=min(300,remaining_budget()))
    assert json.loads(Path('artifacts/improvement-v34/full-host/training-audit.json').read_text())['verified']
Path('v37-route.json').write_text(json.dumps({'related_gate_passed':V37_GATE_PASSED,'route':'new8b' if V37_GATE_PASSED else 'incumbent_v21','elapsed_seconds':time.time()-V37_STARTED,'hidden_labels_used':False},indent=2))


# Name That LLM — Two-stage public reproduction V16

Reproduction of [Nishant Dahal, Two-stage LoRA, part two, version 2](https://www.kaggle.com/code/nishantdahal/two-stage-lora-part-two). Upstream documentation and all credits follow unchanged. The reported 0.620 public score and 0.642 CV belong to the upstream author, not this run. We have not independently reproduced its training CV. Our previous best is 0.427.

Public pretrained assets: [NTL handwriting blend](https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend), version 1, CC BY-SA 4.0. No private synthetic data added. Original blend weights, centered heads and Sinkhorn balancing are retained, including their test-distribution assumptions. Added execution audits and output validation only. All inputs perform actual model inference.

# Two-stage LoRA, part two, LB 0.620

Our [first notebook](https://www.kaggle.com/code/nishantdahal/two-stage-lora-on-the-public-pool) scored 0.592 with two Qwen3 LoRAs trained on 35,000 public passages. This one scores 0.620 with the same recipe. Three things changed. The public pool got bigger and cleaner, two new members joined, and fixed a tokenizer bug that only shows up on Kaggle. Below is what each change did, on the same 5-fold CV split by prompt.

| Member | Trains on | First notebook | This notebook |
|---|---|---|---|
| n-gram SVC | train plus the public pool, fit here | 0.435 | 0.473 |
| Qwen3-14B LoRA, 27 of 40 layers | public pool, then train | 0.569 | 0.611 |
| Qwen2.5-14B LoRA, 32 of 48 layers | public pool, then train | | 0.569 |
| DeBERTa-v3-large, full fine tune | public pool, then train | | 0.546 |
| Blend with Sinkhorn, CV | | 0.593 | **0.642** |
| Public LB, with the centred twin | | 0.592 | **0.620** |

Member rows are balanced accuracy of that member alone, out of fold. The first notebook also had a Qwen3-8B LoRA at 0.569, which is gone now.

Everything the notebook needs is in the dataset [NTL two-stage LoRA part two](https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend). The base models are Kaggle's copies of Qwen3 14B Base and Qwen2.5 14B. Inference only. The commit run on the public test takes about 15 minutes on two T4s.

In [ ]:
import gc, glob, json, os, re, shutil, time

# The scoring run has no internet, so transformers must never try to reach the Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from torch import nn

# Keep fp32 sums in fp16 matmuls. A T4 is otherwise allowed to sum in fp16, which drops digits in a deep model.
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Blend weights. Every member adds weight * log(probability) to the score.
W_SVC = 1.0
LORAS = {  # checkpoint folder, then blend weight, centred twin weight, batch size and the base model to look for
    "lora_qwen3_14b":  dict(w=0.4, twin_w=0.1, bs=8, base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
    "lora_qwen25_14b": dict(w=0.2, twin_w=0.1, bs=8, base=dict(model_type="qwen2", hidden_size=5120, num_hidden_layers=48)),
}
W_DEBERTA = 0.25
TWIN = dict(thresh=0.85, min_size=2, max_size=18)   # topic clusters that get the centred twin

# Column order of every checkpoint's head. Each checkpoint's meta.json is checked against it below.
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def vote(name, P, w, rows=slice(None)):
    """Add one member's weighted log probabilities to the running score."""
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1))
    print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

In [ ]:
from pathlib import Path
assert torch.cuda.device_count() >= 2, 'Dual GPUs required'
assert len(train)==2699 and train.id.is_unique and test.id.is_unique
assert set(train.model.unique()) == set(LABELS) and not pd.isna(y).any()
MEMBER_AUDIT=[]
_original_vote=vote
def vote(name,P,w,rows=slice(None)):
    assert P.ndim==2 and P.shape[1]==K and np.isfinite(P).all()
    assert np.allclose(P.sum(1),1,atol=1e-5)
    MEMBER_AUDIT.append({'member':name,'weight':float(w),'rows':len(P)})
    return _original_vote(name,P,w,rows)
print({'audit':'real inference on every input','devices':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]},flush=True)


## 1. n-gram SVC

The same four TF-IDF views as the first notebook. Raw characters, words with punctuation, word shapes, and small words with every content word replaced by CONTENT. The same LinearSVC with C of 0.5 and balanced class weights. The only change is the pool it fits on, 75,783 public passages instead of 34,940. That alone moved it from 0.435 to 0.473.

In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

SMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]


def shape_text(t):
    """Every run of letters becomes X or x, so only the rhythm is left."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


def small_words_text(t):
    """Keep the small words and the punctuation, and replace every other word by CONTENT."""
    return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\w+|[^\w\s]", t))


pool = pd.read_parquet(find("pool_v4.parquet"))
fit_texts = train.text.astype(str).tolist() + pool.text.tolist()
fit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values]
A, B = [], []
for view, lo, hi, max_features, weight in VIEWS:
    prep = shape_text if view == "shape" else small_words_text if view == "small" else (lambda s: s)
    analyzer = "char" if view == "shape" else "word" if view == "small" else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                max_features=max_features, dtype=np.float32)
    if analyzer == "word":
        opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    vec = TfidfVectorizer(**opts)
    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
    B.append(vec.transform([prep(t) for t in test.text]) * weight)
svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(hstack(A, format="csr"), fit_y)
vote("svc", softmax(svc.decision_function(hstack(B, format="csr"))), W_SVC)
del A, B, svc, pool
gc.collect()

## 2. The pool, the change that mattered most

Most of the gain came from data, not from models. On the old pool of 35,000 passages, four backbones finished at the same 0.569. Qwen3-8B, Qwen3-14B, Qwen2.5-7B and Qwen2.5-14B. The same Qwen3-14B with the same recipe on the new pool finished at 0.611.

You can see it before the host data even comes in. After stage one alone, trained only on public passages, the 14B's head scores 0.572 on the host passages. With the old pool it scored 0.493. The backbone did not change. The pool did.

The pool the 14B read in stage one has 83,827 passages from 26 public datasets. It keeps the filters of the first notebook. One paragraph, 30 to 300 words, no markdown, no duplicates, nothing identical to a train passage, and no punctuation mark that a model never uses in the host data. Two filters are new.

- **One source is out.** A set of claude-fable-5 passages written with a system prompt. They read nothing like the host's Fable passages.
- **Public passages too close to a train passage are out.** 1,153 passages whose content TF-IDF cosine with some train passage is 0.3 or more. Some public datasets answer prompts rebuilt from the training data. If stage one has already seen a host prompt, the out of fold score of stage two is too kind. This filter keeps the CV honest. It does not raise it.

The dataset ships the 75,783 passage version that the SVC fits on. It has the first filter but not the second.

## 3. Two new members

**Qwen2.5-14B LoRA.** The same recipe as the Qwen3 LoRAs, on the first 32 of 48 layers, pooled at layers 3, 5, 8, 11, 16, 21, 27 and 32. Stage one ran on the old 35,000 passage pool. Alone it ties the Qwen3 models at 0.569. Its top prediction agrees with theirs on about 77 percent of passages, which is about as often as they agree with each other. So it is not a new view, and it gets a small weight, 0.2.

**DeBERTa-v3-large, fully fine tuned.** No LoRA here, all the weights train. The same two stages. Stage one is 2 epochs over 61,486 public passages with class weights, at learning rate 0.000015 for the encoder and 0.001 for the head. Stage two is 3 epochs on train at 0.00001 and 0.0005. Max length 384, the mean of the last layer, one linear head. The bigger pool moved it from 0.512 to 0.546 alone.

## 4. LoRA inference, and a bug that only shows up on Kaggle

The inference code is the first notebook's. Keep the first layers, merge the adapter into the weights, average the hidden states at 8 depths without the first token, one LayerNorm per depth, one linear head. Two things are new.

**The end token.** Every passage gets an end token appended, in training and at inference. Our training tokenizer used `<|endoftext|>`, id 151643. Kaggle's copy of Qwen3-14B-Base ships the tokenizer config of the chat model, whose end token is `<|im_end|>`, id 151645. So on Kaggle every passage ended with a token the model never saw there in training, and the 14B's top prediction changed on about 14 percent of passages compared with the same checkpoint run locally. Nothing crashes and nothing warns. The fix is one line. Use the id saved in the checkpoint's meta.json. Kaggle's Qwen2.5-14B does not have this problem. If you run Qwen3-14B-Base from Kaggle Models, print which end token you get.

**T4 numerics and memory.**

- We tell cuBLAS to keep fp32 sums in fp16 matmuls. It costs nothing, so we set it.
- A batch that overflows fp16 is redone in bf16, as before.
- A batch that runs out of memory on the 15 GB T4 is redone one passage at a time.
- There is no try except block around the members. In an earlier submission a handler like that skipped a member that ran out of memory on the T4. The run still finished and scored, a few points lower, and we only found out from the log. A member that breaks should stop the notebook.

In [ ]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    """The attached base model whose config.json matches spec."""
    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f"attach the base model {spec}")


def load_trunk(base_dir, keep, dtype):
    """The first keep decoder layers, split over both T4s when they do not fit on one."""
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, "layer_types", None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    H, I, V = mc.hidden_size, mc.intermediate_size, mc.vocab_size
    hd = getattr(mc, "head_dim", H // mc.num_attention_heads)
    q, kv = mc.num_attention_heads * hd, mc.num_key_value_heads * hd
    gib = 2 * (V * H + keep * (2 * H * q + 2 * H * kv + 3 * H * I)) / 2 ** 30
    if gib > 13 and torch.cuda.device_count() > 1:
        mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
    else:
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
    model.norm = nn.Identity()   # the checkpoints were trained without the final norm
    print(f"trunk of {keep} layers, {gib:.1f} GiB of weights", flush=True)
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    """W becomes W + B A alpha / r for every adapted linear layer."""
    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
    scale = acfg["lora_alpha"] / acfg["r"]
    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if ".lora_A." not in k:
            continue
        name = k.split(".lora_A.")[0]
        lin = modules[name[name.index("layers."):]]
        Bm = sd[k.replace(".lora_A.", ".lora_B.")]
        delta = (Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale
        lin.weight.add_(delta.to(lin.weight.dtype))


@torch.no_grad()
def lora_features(ckpt_dir, base_dir, texts, bs):
    """Pooled features Z, one row per passage, and the checkpoint's heads."""
    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
    cfg = meta["cfg"]
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, cfg["keep"], dtype)
    merge_lora(model, os.path.join(ckpt_dir, "adapter"))
    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
    layers, H = cfg["layers"], model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
    # The end token from training, not this tokenizer's. Kaggle's Qwen3-14B-Base carries the chat model's end token.
    eos_id = cfg.get("eos_id", tok.eos_token_id)
    print(f"end token {eos_id}, this tokenizer's is {tok.eos_token_id}", flush=True)
    ids = [x + [eos_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together means less padding
    Z = np.zeros((len(ids), H * len(layers)), np.float32)

    def embed(idx):
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow, so redo this batch in bf16
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            model.to(dtype)
        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
        m[:, 0] = 0   # the first token soaks up attention whatever it is, so leave it out of the average
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()

    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        try:
            embed(idx)
        except torch.cuda.OutOfMemoryError:   # the longest batch can overflow a 15 GB T4, so go one passage at a time
            torch.cuda.empty_cache()
            for i in idx:
                embed([i])
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}

## 5. Centred twin

As in the first notebook. Cluster the test passages by content TF-IDF. In every cluster of 2 to 18 passages, subtract the cluster mean from each LoRA's features and read the rest with the centred head, at weight 0.1 per LoRA. It is not in the CV numbers above, because CV has no test clusters.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

if len(test) >= 2:
    Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
    D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    groups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance")
    del D
else:
    groups = np.ones(len(test), dtype=int)
size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
print(f"topic clusters: {int((size == 1).sum())} passages alone, {int(twin_rows.sum())} in clusters of 2 to 18", flush=True)

for ckpt_name, mc in LORAS.items():
    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z, heads = lora_features(ckpt, find_base(mc["base"]), test.text.tolist(), mc["bs"])
    vote(ckpt_name, softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), mc["w"])
    if twin_rows.any():
        Zc = Z.copy()
        for g in np.unique(groups[twin_rows]):
            Zc[groups == g] -= Zc[groups == g].mean(0)
        centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
        vote(ckpt_name + " twin", softmax(centred), mc["twin_w"], rows=twin_rows)
    del Z
    gc.collect()

## 6. DeBERTa

The checkpoint is copied to local disk before loading. Loading it straight from the dataset mount took 16 minutes in our runs. The copy plus the load take 25 seconds.

In [ ]:
ck = os.path.dirname(find("deberta_v3_large/meta.json"))
if ck.startswith("/kaggle/input"):   # one long copy to local disk beats many small reads from the dataset mount
    shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
    ck = "/tmp/deberta_v3_large"
meta = json.load(open(os.path.join(ck, "meta.json")))
assert meta["labels"] == LABELS, "label order differs from the checkpoint's head"
etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
head = load_file(os.path.join(ck, "head.safetensors"))
W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
ids = etok(test.text.tolist(), truncation=True, max_length=meta["max_len"])["input_ids"]
order = np.argsort([len(x) for x in ids], kind="stable")
Z = np.zeros((len(ids), enc.config.hidden_size), np.float32)
with torch.inference_mode():
    for s in range(0, len(order), 32):
        idx = order[s:s + 32]
        bt = etok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to("cuda:0")
        m = bt["attention_mask"].unsqueeze(-1).float()
        Z[idx] = ((enc(**bt).last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
vote("deberta", softmax(Z @ W.T + b), W_DEBERTA)
del enc
gc.collect()
torch.cuda.empty_cache()

## 7. Blend and Sinkhorn

A weighted sum of log probabilities. SVC 1.0, Qwen3-14B 0.4, Qwen2.5-14B 0.2, DeBERTa 0.25. These are round numbers chosen on out of fold predictions. Fitting them finer on 2,699 passages mostly fits noise. Then Sinkhorn balancing to equal class totals as in the first notebook, 300 iterations, skipped on the 3 row public test.

In [ ]:
if not V37_GATE_PASSED:
    exec('\nimport hashlib\nV21_ASSET_HASHES={\'full/adapter/adapter_model.safetensors\': \'7d07a6e4f7cd80adf46d7918b33c40ce9a019e96721e4aa667f03b6b42254c5c\', \'full/adapter/adapter_config.json\': \'b052b98c261b28ea69614dec8f0740dbfb3f9469d96ba4887e8aefec8a2b1feb\', \'full/readout.npz\': \'d289c99281001a208bb77da49d03b6ae0a966c5679a36aa94b6d450aed8a57c8\', \'full/training_result.json\': \'e892f25e65b05efdceac9f48999c139f8a331c19bee01cef33153397a1656e22\', \'full/readout_result.json\': \'04c1043c4820c0fdcef0699b57f2a060cf9eb0581b4d4fd1c0ad60cc893e7169\'}\nroots=[Path(p).parent for p in glob.glob(\'/kaggle/input/**/plan.json\',recursive=True)\n       if json.loads(Path(p).read_text()).get(\'experiment\')==\'improvement-v21\']\nassert len(roots)==1,roots\nv21_root=roots[0]\nfor relative,digest in V21_ASSET_HASHES.items():\n    assert hashlib.sha256((v21_root/relative).read_bytes()).hexdigest()==digest,relative\nsource=\'"""Feature extraction and portable logistic readout for the new v21 member."""\\nimport gc\\nimport json\\nfrom pathlib import Path\\nimport time\\nimport warnings\\nimport numpy as np\\nimport torch\\nfrom scipy.special import log_softmax\\n\\n\\nclass ContrastiveReadout:\\n    def __init__(self, model_dir, adapter, layers=(12,24,36), max_length=256, batch_size=8):\\n        self.model_dir, self.adapter = model_dir, adapter\\n        self.layers, self.max_length, self.batch_size = list(layers), max_length, batch_size\\n\\n    def features(self, texts):\\n        from transformers import AutoModel, AutoTokenizer\\n        from peft import PeftModel\\n        tokenizer = AutoTokenizer.from_pretrained(self.model_dir,local_files_only=True)\\n        tokenizer.padding_side=\\\'right\\\'\\n        if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token\\n        tokens=tokenizer(list(texts),truncation=True,max_length=self.max_length-1,\\n                         add_special_tokens=False)[\\\'input_ids\\\']\\n        tokens=[ids+[tokenizer.eos_token_id] for ids in tokens]\\n        order=np.argsort([len(ids) for ids in tokens],kind=\\\'stable\\\')\\n        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction=False\\n        model=AutoModel.from_pretrained(self.model_dir,torch_dtype=torch.float16,\\n                                        attn_implementation=\\\'sdpa\\\',local_files_only=True)\\n        model=PeftModel.from_pretrained(model,self.adapter,is_trainable=False).cuda().eval()\\n        features=np.empty((len(texts),len(self.layers),model.config.hidden_size),dtype=np.float32)\\n        started=time.monotonic()\\n        with torch.inference_mode():\\n            for start in range(0,len(texts),self.batch_size):\\n                ids=order[start:start+self.batch_size]\\n                batch={k:v.cuda() for k,v in tokenizer.pad({\\\'input_ids\\\':[tokens[i] for i in ids]},\\n                                                          padding=True,return_tensors=\\\'pt\\\').items()}\\n                with torch.autocast(\\\'cuda\\\',dtype=torch.float16):\\n                    states=model(**batch,output_hidden_states=True,use_cache=False).hidden_states\\n                mask=batch[\\\'attention_mask\\\'].clone();mask[:,0]=0\\n                for j,layer in enumerate(self.layers):\\n                    h=states[layer].float()\\n                    features[ids,j]=((h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)).cpu().numpy()\\n                del states\\n                if start%400==0:\\n                    print(json.dumps({\\\'event\\\':\\\'v21_features\\\',\\\'completed\\\':min(start+len(ids),len(texts)),\\n                                      \\\'total\\\':len(texts),\\\'seconds\\\':time.monotonic()-started}),flush=True)\\n        del model;gc.collect();torch.cuda.empty_cache()\\n        assert np.isfinite(features).all()\\n        return features\\n\\n    @staticmethod\\n    def predict(features, path):\\n        with np.load(path,allow_pickle=False) as state:\\n            assert features.shape[1:]==(len(state[\\\'layers\\\']),state[\\\'mean\\\'].shape[-1])\\n            scores=[]\\n            for j in range(features.shape[1]):\\n                X=(features[:,j].astype(np.float64)-state[\\\'mean\\\'][j])/state[\\\'scale\\\'][j]\\n                scores.append(log_softmax(X@state[\\\'coef\\\'][j].T+state[\\\'intercept\\\'][j],axis=1))\\n            labels=state[\\\'labels\\\'].copy()\\n        scores=np.mean(scores,axis=0)\\n        assert np.isfinite(scores).all()\\n        return labels,scores\\n\\n    def fit(self,features,targets,path,C=0.02,max_iter=1000,seed=1901):\\n        from sklearn.preprocessing import StandardScaler\\n        from sklearn.linear_model import LogisticRegression\\n        from sklearn.exceptions import ConvergenceWarning\\n        means,scales,coefs,intercepts,predictions=[],[],[],[],[]\\n        labels=np.sort(np.unique(targets))\\n        for j,layer in enumerate(self.layers):\\n            scaler=StandardScaler()\\n            X=scaler.fit_transform(features[:,j].astype(np.float64))\\n            clf=LogisticRegression(C=C,max_iter=max_iter,random_state=seed)\\n            with warnings.catch_warnings():\\n                warnings.simplefilter(\\\'error\\\',ConvergenceWarning)\\n                clf.fit(X,targets)\\n            assert np.array_equal(labels,clf.classes_)\\n            means.append(scaler.mean_);scales.append(scaler.scale_)\\n            coefs.append(clf.coef_);intercepts.append(clf.intercept_)\\n            predictions.append(clf.predict_log_proba(X))\\n        np.savez_compressed(path,labels=labels.astype(str),layers=self.layers,mean=np.stack(means),\\n                            scale=np.stack(scales),coef=np.stack(coefs),intercept=np.stack(intercepts))\\n        recovered_labels,recovered_scores=self.predict(features,path)\\n        assert np.array_equal(recovered_labels,labels)\\n        np.testing.assert_allclose(recovered_scores,np.mean(predictions,axis=0),atol=1e-8,rtol=1e-8)\\n        return {\\\'fit_rows\\\':len(targets),\\\'layers\\\':self.layers,\\\'labels\\\':labels.tolist(),\\n                \\\'portable_readout_check\\\':\\\'passed\\\',\\\'C\\\':C,\\\'dtype\\\':\\\'float64 for scaler and LR\\\'}\\n\\n\\ndef main():\\n    import hashlib\\n    import pandas as pd\\n    plan=json.loads(Path(\\\'plan.json\\\').read_text())\\n    frame=pd.read_csv(plan[\\\'host_path\\\'],dtype={\\\'id\\\':str})\\n    component=ContrastiveReadout(plan[\\\'model_dir\\\'],\\\'full/adapter\\\',layers=plan[\\\'readout\\\'][\\\'layers\\\'],\\n                                max_length=plan[\\\'training_parameters\\\'][\\\'max_length\\\'],\\n                                batch_size=plan[\\\'readout\\\'][\\\'batch_size\\\'])\\n    features=component.features(frame.text.tolist())\\n    np.save(\\\'full/train_features.npy\\\',features)\\n    result=component.fit(features,frame.model.to_numpy(),\\\'full/readout.npz\\\',\\n                          C=plan[\\\'readout\\\'][\\\'C\\\'],max_iter=plan[\\\'readout\\\'][\\\'max_iter\\\'],seed=plan[\\\'readout\\\'][\\\'seed\\\'])\\n    result[\\\'artifacts_sha256\\\']={str(p):hashlib.sha256(p.read_bytes()).hexdigest()\\n                                for p in [Path(\\\'full/adapter/adapter_model.safetensors\\\'),Path(\\\'full/readout.npz\\\'),Path(\\\'full/train_features.npy\\\')]}\\n    Path(\\\'full/readout_result.json\\\').write_text(json.dumps(result,indent=2))\\n    print(json.dumps(result),flush=True)\\n\\n\\nif __name__==\\\'__main__\\\': main()\\n\'\nassert hashlib.sha256(source.encode()).hexdigest()==\'820fc727fd549736b97df9888dc51065fdc89a653c8ac90d689e370aa244c417\'\nPath(\'component_v21.py\').write_text(source)\nfrom component_v21 import ContrastiveReadout\nv21_model_dir=find_base(dict(model_type=\'qwen3\',hidden_size=2560,num_hidden_layers=36))\ncomponent=ContrastiveReadout(v21_model_dir,str(v21_root/\'full/adapter\'),layers=[12, 24, 36],\n                            max_length=256,batch_size=8)\nv21_features=component.features(test.text.tolist())\nv21_labels,v21_scores=component.predict(v21_features,v21_root/\'full/readout.npz\')\nassert set(v21_labels)==set(LABELS) and len(set(v21_labels))==K\nv21_order=[list(v21_labels).index(label) for label in LABELS]\nv21_scores=v21_scores[:,v21_order]\nassert v21_scores.shape==LOGIT.shape and np.isfinite(v21_scores).all()\nV21_INCUMBENT_LOGIT=LOGIT.copy()\nvote(\'contrastive_v21\',softmax(v21_scores),0.15)\nnp.savez_compressed(\'v21_scores.npz\',ids=test.id.to_numpy(dtype=str),labels=np.asarray(LABELS),\n                    incumbent_scores=V21_INCUMBENT_LOGIT,component_scores=v21_scores,combined_scores=LOGIT)\ndel component,v21_features\ngc.collect();torch.cuda.empty_cache()\n',globals())
    V37_SELECTED_MEMBER='contrastive_v21'
else:
    import importlib.util
    module_path=V37_FULL_ROOT/'component_v34.py'
    spec=importlib.util.spec_from_file_location('v37_component',module_path)
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    plan=json.loads((V37_FULL_ROOT/'plan.json').read_text())
    assert hashlib.sha256(module_path.read_bytes()).hexdigest()==plan['source_sha256']['component_v34.py']
    component=module.ContrastiveReadout(find_base(dict(model_type='qwen3',hidden_size=4096,num_hidden_layers=36)),str(V37_FULL_ROOT/'full/adapter'),layers=[12,24,36],max_length=plan['training_parameters']['max_length'],batch_size=8)
    features=component.features(test.text.tolist())
    labels,scores=component.predict(features,V37_FULL_ROOT/'full/readout.npz')
    assert set(labels)==set(LABELS) and len(labels)==K
    scores=scores[:,[list(labels).index(label) for label in LABELS]]
    assert scores.shape==LOGIT.shape and np.isfinite(scores).all()
    vote('ce8b_v37',softmax(scores),0.15)
    np.savez_compressed('v37_component_scores.npz',ids=test.id.to_numpy(dtype=str),labels=np.asarray(LABELS),scores=scores)
    del component,features;gc.collect();torch.cuda.empty_cache()
    V37_SELECTED_MEMBER='ce8b_v37'


In [ ]:
def sinkhorn(P, iters=300):
    """One scale per class, so that every class gets an equal share of the probability. Rows are renormalised."""
    c = np.ones(K)
    target = np.full(K, len(P) / K)
    for _ in range(iters):
        R = P * c
        R /= R.sum(1, keepdims=True)
        c *= target / R.sum(0)
    R = P * c
    return R / R.sum(1, keepdims=True)


P = softmax(LOGIT)
if len(test) >= 50:   # skip on the 3 row public test.csv
    P = sinkhorn(P)
sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
assert len(sub) == len(test) and sub["model"].isin(LABELS).all()
sub.to_parquet("submission.parquet", index=False)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")

## CV says 0.642, LB says 0.620

The first notebook lost almost nothing between CV and LB, 0.593 against 0.592. This one loses 2.2 points. What changed between the two is mostly the pool. Our CV only tests on host passages, but the members learned much of what they know from public passages written with other settings. Some of that seems not to carry over to the hidden test. We have guesses and no clean answer yet. Part of the gap is plain noise too, since the public LB is only a slice of the test.

## What did not help

- **More parameters.** See section 2.
- **Averaging seeds**, or averaging the LoRA weights of several runs. 0.1 points at most.
- **A Gemma-4-12B backbone** with the same recipe.
- **Gates for one family**, for example a separate judge just for the Claude and GLM passages.
- **At most one passage per model inside a topic cluster.** Better in simulation, worse on the LB.
- **More stage two epochs**, or public passages mixed into stage two.
- **Stage two on short windows** cut from the host passages. Short passages are hard for every member, and the windows did not change that.
- **A stacker** instead of fixed weights.

None of it helped the two hardest pairs, `glm-5.3` against `glm-5.3-flash` and `gemini-3.7-flash` against `gemini-3.8-flash`. Every member gets stuck at about the same place on them.

## Credits

Public passages, CC BY 4.0 unless another license is given.

- [seyitkaangunes/ntl-extra-api-passages](https://www.kaggle.com/datasets/seyitkaangunes/ntl-extra-api-passages)
- [willrice/name-that-llm-openrouter-harvest](https://www.kaggle.com/datasets/willrice/name-that-llm-openrouter-harvest)
- [lxdlxdlxd/ntl-synth-generations-v1](https://www.kaggle.com/datasets/lxdlxdlxd/ntl-synth-generations-v1)
- [prvsiyan/prvsiyans-extended-name-that-llm-dataset](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset)
- [alperen5252525/ntl-vertex-gemini-proxy-gpt-generations](https://www.kaggle.com/datasets/alperen5252525/ntl-vertex-gemini-proxy-gpt-generations)
- [haileycheng/name-that-llm-same-prompt-passages](https://www.kaggle.com/datasets/haileycheng/name-that-llm-same-prompt-passages)
- [kaplinskiy/ntl-gpt-proxy-passages](https://www.kaggle.com/datasets/kaplinskiy/ntl-gpt-proxy-passages)
- [lucasboesen/ntl-free-proxy-generations](https://www.kaggle.com/datasets/lucasboesen/ntl-free-proxy-generations)
- [abdullah1afify/ntl-generations](https://www.kaggle.com/datasets/abdullah1afify/ntl-generations)
- [dophuchung/ntl-proxy-generations](https://www.kaggle.com/datasets/dophuchung/ntl-proxy-generations)
- [ziyuzhou1207/ntl-zz-proxy-generations](https://www.kaggle.com/datasets/ziyuzhou1207/ntl-zz-proxy-generations)
- [keshavatearth/ntl-gpt-new-prompts-v1](https://www.kaggle.com/datasets/keshavatearth/ntl-gpt-new-prompts-v1)
- [michaelsemenoff/ntl-own-extra-v1](https://www.kaggle.com/datasets/michaelsemenoff/ntl-own-extra-v1)
- [jadkutbi/ntl-own-proxy-generations](https://www.kaggle.com/datasets/jadkutbi/ntl-own-proxy-generations)
- [pardheev/ntl-proxy-extra-passages](https://www.kaggle.com/datasets/pardheev/ntl-proxy-extra-passages)
- [abhinavsaipodugu/ntl-proxy-gpt-gemini-passages](https://www.kaggle.com/datasets/abhinavsaipodugu/ntl-proxy-gpt-gemini-passages)
- [zakirkhanaleemi/name-that-llm-benchmarks-proxy-generations](https://www.kaggle.com/datasets/zakirkhanaleemi/name-that-llm-benchmarks-proxy-generations)
- [zbigniewkirchner/ntl-extra-gpt-passages](https://www.kaggle.com/datasets/zbigniewkirchner/ntl-extra-gpt-passages)
- [arxitekton/ntl-claude-gen-v1](https://www.kaggle.com/datasets/arxitekton/ntl-claude-gen-v1)
- [arxitekton/ntl-claude-gen-v2](https://www.kaggle.com/datasets/arxitekton/ntl-claude-gen-v2)
- [maniceet/name-that-llm-openrouter-openai-generations](https://www.kaggle.com/datasets/maniceet/name-that-llm-openrouter-openai-generations)
- [dedruck/ntl-proxy-gen](https://www.kaggle.com/datasets/dedruck/ntl-proxy-gen)
- [prvsiyan/ntl-own-prompt-generations-v1](https://www.kaggle.com/datasets/prvsiyan/ntl-own-prompt-generations-v1), CC BY-SA 4.0
- [drjcfmd/ntl-generated-v2](https://www.kaggle.com/datasets/drjcfmd/ntl-generated-v2), CC BY-SA 4.0
- [illiadolenko/ntl-generated-corpus-v1](https://www.kaggle.com/datasets/illiadolenko/ntl-generated-corpus-v1), CC0
- [aswathvanan/ntl-proxy-passages](https://www.kaggle.com/datasets/aswathvanan/ntl-proxy-passages), CC BY-SA 4.0, only in stage one of the Qwen3-14B LoRA

Ideas and recipes.

- The LoRA24 recipe from [lightningv08/ntl-lora24-fivefold-training](https://www.kaggle.com/code/lightningv08/ntl-lora24-fivefold-training) and [lucasboesen/name-that-llm-train-lora24-s7](https://www.kaggle.com/code/lucasboesen/name-that-llm-train-lora24-s7)
- The style aware n-gram views from [yangkuangou](https://www.kaggle.com/yangkuangou)'s notebook
- Adaptive centering, the centred twin and Sinkhorn balancing from [lucasboesen/lora-x-adaptive-centering](https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering)

Models. Qwen3-14B-Base and Qwen2.5-14B by the Qwen team, Apache 2.0. DeBERTa-v3-large by Microsoft, MIT.

In [ ]:
assert set(m['member'] for m in MEMBER_AUDIT)>={'svc','lora_qwen3_14b','lora_qwen25_14b','deberta',V37_SELECTED_MEMBER}
assert np.isfinite(LOGIT).all() and np.isfinite(P).all() and np.allclose(P.sum(1),1,atol=1e-5)
assert sub.columns.tolist()==['id','model'] and sub.id.tolist()==test.id.astype(str).tolist()
assert sub.id.is_unique and sub.model.isin(LABELS).all() and pd.read_parquet('submission.parquet').equals(sub)
audit={'experiment':'v37','route':'new8b' if V37_GATE_PASSED else 'incumbent_v21','related_gate':V37_GATE,'rows':len(test),'members':MEMBER_AUDIT,'elapsed_seconds':time.time()-V37_STARTED,'claimed_public_gain':None,'hidden_labels_used':False,'selection':'fixed training-only related gate; no hidden score selection','attribution':'Credited V16 ensemble: Nishant Dahal and contributors. Qwen backbones; established LoRA/SupCon/QLoRA methods.'}
Path('v37-audit.json').write_text(json.dumps(audit,indent=2));print(json.dumps(audit),flush=True)
